# Level 5a – Umbrella sampling and WHAM

**Idea:** instead of pulling continuously, hold the ligand at a series of fixed distances (windows) with a harmonic restraint and let everything else equilibrate. Each window samples a narrow range of the pulling coordinate. WHAM (weighted histogram analysis) removes the known restraint energy and joins the windows into one free-energy profile, the potential of mean force (PMF).

**Setup:** starting structures are taken from pull 1 of Level 4, about every 0.1 nm along the pulling coordinate (roughly 20 windows). Each window runs 100 ps of equilibration and `PROD_NS` of production with a 1000 kJ/mol/nm² restraint. The C-alpha restraints from Level 4 stay on, so this is the PMF for a restrained protein.

**Time on a T4:** roughly 15–20 minutes per window at 1 ns, so 5–6 hours for all windows. It resumes window by window, and `WINDOWS` lets you run a subset per session.

**Requires:** Level 4 finished (at least pull 1).

## 0. Setup
Run these cells at the start of every Colab session. The install is skipped if the environment already exists in this session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /content/mdenv/bin/gmx ]; then
  ./bin/micromamba create -y -q -p /content/mdenv -c conda-forge python=3.12 \
      "gromacs=*=nompi_cuda*" acpype ambertools openbabel 

fi
/content/mdenv/bin/gmx --version 2>/dev/null | grep -E "GROMACS version|GPU support"

In [ ]:
!pip install -q py3Dmol MDAnalysis openmm 
!pip install -q pdbfixer || pip install -q git+https://github.com/openmm/pdbfixer.git

In [ ]:
# Make the conda environment's programs visible to this notebook
import os, json, subprocess, shutil, glob
ENV = "/content/mdenv"
r = subprocess.run(["/content/bin/micromamba", "run", "-p", ENV, "python", "-c",
                    "import os, json; print(json.dumps(dict(os.environ)))"], capture_output=True, text=True)
try:
    activated = json.loads(r.stdout.strip().splitlines()[-1])
except Exception:
    activated = {}
for key in ["PATH", "AMBERHOME", "BABEL_LIBDIR", "BABEL_DATADIR", "GMXDATA", "GMXBIN", "GMXLDLIB",
            "GMXMAN", "GROMACS_DIR", "CONDA_PREFIX", "PLUMED_KERNEL", "PLUMED_ROOT"]:
    if key in activated:
        os.environ[key] = activated[key]
if f"{ENV}/bin" not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{ENV}/bin:" + os.environ["PATH"]
os.environ.setdefault("AMBERHOME", ENV)
os.environ.setdefault("GMXDATA", f"{ENV}/share/gromacs")
os.environ["GMXLIB"] = os.environ["GMXDATA"] + "/top"

for tool in ["gmx", "acpype", "obabel"]:
    print(f"{tool:12s}", shutil.which(tool) or "NOT FOUND")

In [ ]:
# Results live on Google Drive so they survive disconnects (set USE_DRIVE = False to keep them on the VM)
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/md_results"
else:
    OUTPUT_ROOT = "/content/md_results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("Results folder:", OUTPUT_ROOT)

In [ ]:
%%writefile /content/md_pipeline.py
#!/usr/bin/env python3
"""
md_pipeline.py  -  Reusable protein-ligand MD pipeline (GROMACS + local GPU)
==============================================================================

Designed to run from a Google Colab notebook connected to a LOCAL runtime, so
every GROMACS call executes on your own machine and uses your own NVIDIA GPU.
It also works as a plain command-line script.

Default example: T4 lysozyme L99A with benzene (PDB 181L, ligand resname BNZ),
a small, well-studied protein-ligand complex (~2.6k protein atoms).

Pipeline
--------
 00_input           downloaded / copied structure
 01_ligand          ligand extraction, protonation (Open Babel), GAFF2 + AM1-BCC (acpype)
 02_protein         protein cleaning (PDBFixer, optional)
 03_system          pdb2gmx (AMBER99SB-ILDN / TIP3P), complex, box, water, ions
 04_em              energy minimisation
 05_nvt             NVT equilibration (protein + ligand heavy atoms restrained)
 06_npt             NPT equilibration (restrained)
 07_production      unrestrained production MD (resumable from checkpoint)
 08_analysis        PBC-corrected/fitted trajectory, RMSD, RMSF, Rg, contacts,
                    H-bonds, energies -> .xvg + .csv + .png + summary.json
 09_visualization   complex-only trajectory + PDBs + ChimeraX scripts (.cxc)
 logs               full stdout/stderr of every command

Re-running is safe: finished stages are skipped (set overwrite=True to redo),
and an interrupted production run resumes from md.cpt.

One-time setup on your LOCAL machine (Linux, or Windows via WSL2; NVIDIA driver installed)
-------------------------------------------------------------------------------
  conda create -n md -c conda-forge python=3.11 "gromacs=*=nompi_cuda*" \
        acpype ambertools openbabel pdbfixer numpy matplotlib py3dmol jupyter
  conda activate md
  gmx --version | grep "GPU support"        # should print: CUDA

  # start Jupyter so Colab can connect to it
  jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' \
        --port=8888 --NotebookApp.port_retries=0
  #   (Notebook >= 7:  --ServerApp.allow_origin=... --ServerApp.port_retries=0)
  # In Colab: Connect (top right) > "Connect to a local runtime" > paste the
  # http://localhost:8888/?token=... URL printed by Jupyter.

Use in a Colab cell
-------------------
  import sys; sys.path.append("/path/to/folder/with/this/script")
  from md_pipeline import Config, run_pipeline, open_in_chimerax, show_in_notebook

  cfg = Config(name="t4l_benzene", pdb_id="181L", ligand_resname="BNZ",
               ligand_charge=0, prod_ns=10)
  run_pipeline(cfg)
  open_in_chimerax(cfg)          # launches ChimeraX on your desktop
  show_in_notebook(cfg)          # quick 3D view inside the notebook (py3Dmol)

Your own system
---------------
  Config(name="my_complex", pdb_file="complex.pdb", ligand_resname="XYZ",
         ligand_charge=-1)
  # Recommended for drug-like ligands (PDB files carry no bond orders):
  Config(name="my_complex", pdb_file="protein.pdb",
         ligand_file="ligand_docked_withH.sdf", ligand_charge=0)
  # the ligand file must already be posed in the protein's coordinate frame
  # and contain all hydrogens. Apo protein: leave ligand_resname/ligand_file None.

Command line
------------
  python md_pipeline.py --pdb-id 181L --ligand-resname BNZ --ligand-charge 0 \
         --name t4l_benzene --prod-ns 10 --chimerax
  python md_pipeline.py --config my_system.json
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from dataclasses import asdict, dataclass, fields
from pathlib import Path

LIG = "LIG"  # residue / molecule name used for the ligand everywhere


# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    # --- system ---
    name: str = "t4l_benzene"
    pdb_id: str | None = "181L"          # downloaded from RCSB if pdb_file is None
    pdb_file: str | None = None          # local PDB (protein, or protein+ligand)
    ligand_resname: str | None = "BNZ"   # ligand residue name inside the PDB
    ligand_file: str | None = None       # optional SDF/MOL2 with H (overrides resname)
    ligand_chain: str | None = None      # pick ligand copy from this chain
    ligand_charge: int | None = 0        # net charge; None lets acpype guess
    protein_chains: list[str] | None = None  # e.g. ["A"]; None = all chains
    ph: float = 7.4                      # used when protonating a PDB ligand
    fix_protein: bool = True             # add missing heavy atoms with PDBFixer

    # --- force field ---
    forcefield: str = "amber99sb-ildn"
    water: str = "tip3p"
    ligand_ff: str = "gaff2"
    charge_method: str = "bcc"           # AM1-BCC
    positive_ion: str = "NA"
    negative_ion: str = "CL"

    # --- box / conditions ---
    box_type: str = "dodecahedron"
    box_distance: float = 1.0            # nm, solute-box distance
    salt_conc: float = 0.15              # M
    temperature: float = 300.0           # K
    pressure: float = 1.0                # bar

    # --- lengths ---
    dt: float = 0.002                    # ps
    em_steps: int = 50000
    nvt_ps: float = 100.0
    npt_ps: float = 100.0
    prod_ns: float = 10.0
    traj_interval_ps: float = 10.0       # production frame spacing

    # --- execution ---
    output_root: str = "md_results"
    gmx: str = "gmx"
    use_gpu: bool = True
    gpu_id: str | None = None            # e.g. "0"
    ntomp: int | None = None             # OpenMP threads; None = GROMACS decides
    maxwarn: int = 0
    overwrite: bool = False
    checkpoint_min: float = 5.0          # mdrun checkpoint interval (minutes); short = safe resume
    seed: int = -1                       # velocity seed for NVT (-1 = random); set for replicas
    seed: int = -1                       # velocity + thermostat seed; -1 = random each run

    # --- visualisation ---
    viewer_stride: int = 1               # keep every n-th frame in the viewer trajectory
    chimera_pdb_stride: int = 5          # every n-th frame in the Chimera multi-model PDB (keeps it small)
    chimerax_path: str | None = None     # auto-detected if None
    launch_chimerax: bool = False        # open ChimeraX automatically at the end

    @property
    def has_ligand(self) -> bool:
        return bool(self.ligand_file or self.ligand_resname)

    @classmethod
    def from_json(cls, path: str) -> "Config":
        data = json.loads(Path(path).read_text())
        known = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in data.items() if k in known})


class Paths:
    def __init__(self, cfg: Config):
        self.root = Path(cfg.output_root).expanduser().resolve() / cfg.name
        self.inp = self.root / "00_input"
        self.lig = self.root / "01_ligand"
        self.prot = self.root / "02_protein"
        self.sys = self.root / "03_system"
        self.em = self.root / "04_em"
        self.nvt = self.root / "05_nvt"
        self.npt = self.root / "06_npt"
        self.md = self.root / "07_production"
        self.ana = self.root / "08_analysis"
        self.viz = self.root / "09_visualization"
        self.logs = self.root / "logs"

    def make(self):
        for p in vars(self).values():
            Path(p).mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------- #
# Command runner
# --------------------------------------------------------------------------- #
class Runner:
    def __init__(self, cfg: Config, paths: Paths):
        self.cfg, self.p = cfg, paths
        self.env = os.environ.copy()
        self.env["GMX_MAXBACKUP"] = "-1"  # no #backup# files cluttering the folders

    def run(self, cmd, step, cwd=None, stdin=None, check=True) -> int:
        cmd = [str(c) for c in cmd]
        log = self.p.logs / f"{step}.log"
        print(f"    $ {' '.join(cmd)}")
        with open(log, "a") as fh:
            fh.write(f"\n### {time.ctime()}  cwd={cwd}\n$ {' '.join(cmd)}\n")
            if stdin:
                fh.write(f"(stdin: {stdin!r})\n")
            fh.flush()
            r = subprocess.run(cmd, cwd=cwd, input=stdin, text=True, env=self.env,
                               stdout=fh, stderr=subprocess.STDOUT)
        if check and r.returncode != 0:
            tail = "".join(log.read_text(errors="replace").splitlines(True)[-40:])
            raise RuntimeError(f"Step '{step}' failed (exit {r.returncode}). "
                               f"Full log: {log}\n----- last lines -----\n{tail}")
        return r.returncode

    def gmx(self, args, step, **kw) -> int:
        return self.run([self.cfg.gmx, *args], step, **kw)


def banner(msg):
    print(f"\n=== {msg} " + "=" * max(0, 70 - len(msg)))


def done(path: Path, cfg: Config) -> bool:
    if path.exists() and not cfg.overwrite:
        print(f"    already done ({path.name}) - skipping")
        return True
    return False


# --------------------------------------------------------------------------- #
# Environment checks
# --------------------------------------------------------------------------- #
def check_environment(cfg: Config) -> dict:
    banner("Checking environment")
    info = {}
    if not shutil.which(cfg.gmx):
        raise RuntimeError(f"'{cfg.gmx}' not found on PATH. Activate the conda env "
                           "that has GROMACS (see the header of this script).")
    out = subprocess.run([cfg.gmx, "--version"], capture_output=True, text=True).stdout
    ver = re.search(r"GROMACS version:\s*(\S+)", out)
    gpu = re.search(r"GPU support:\s*(.+)", out)
    info["gromacs_version"] = ver.group(1) if ver else "unknown"
    info["gromacs_gpu_support"] = gpu.group(1).strip() if gpu else "unknown"
    print(f"    GROMACS {info['gromacs_version']}  (GPU support: {info['gromacs_gpu_support']})")

    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                              "--format=csv,noheader"], capture_output=True, text=True)
        info["gpus"] = [l.strip() for l in smi.stdout.splitlines() if l.strip()]
    except FileNotFoundError:
        info["gpus"] = []
    for g in info["gpus"]:
        print(f"    GPU: {g}")

    if cfg.use_gpu:
        if "disabled" in info["gromacs_gpu_support"].lower():
            print("    WARNING: this GROMACS build has no GPU support -> running on CPU.\n"
                  "             Install a CUDA build: conda install -c conda-forge 'gromacs=*=nompi_cuda*'")
            cfg.use_gpu = False
        elif not info["gpus"]:
            print("    WARNING: nvidia-smi found no GPU -> running on CPU. Local runtime: is Colab "
                  "connected to it? Colab cloud: Runtime > Change runtime type > GPU.")
            cfg.use_gpu = False

    if cfg.has_ligand:
        for tool in ("acpype", "obabel"):
            if not shutil.which(tool):
                raise RuntimeError(f"'{tool}' not found. conda install -c conda-forge acpype ambertools openbabel")
    return info


# --------------------------------------------------------------------------- #
# Structure preparation
# --------------------------------------------------------------------------- #
def get_structure(cfg: Config, P: Paths) -> Path:
    banner("Input structure")
    dst = P.inp / "input.pdb"
    if done(dst, cfg):
        return dst
    if cfg.pdb_file:
        shutil.copy(Path(cfg.pdb_file).expanduser(), dst)
        print(f"    copied {cfg.pdb_file}")
    elif cfg.pdb_id:
        url = f"https://files.rcsb.org/download/{cfg.pdb_id.upper()}.pdb"
        print(f"    downloading {url}")
        urllib.request.urlretrieve(url, dst)
    else:
        raise ValueError("Give either pdb_id or pdb_file")
    return dst


def _first_model(lines):
    out = []
    for l in lines:
        if l.startswith("ENDMDL"):
            break
        out.append(l)
    return out


def _altloc_ok(line):
    return len(line) > 16 and line[16] in (" ", "A")


def _no_altloc(line):
    return line[:16] + " " + line[17:]


def prepare_protein(cfg: Config, P: Paths, src: Path) -> Path:
    banner("Protein")
    raw = P.prot / "protein_raw.pdb"
    clean = P.prot / "protein_clean.pdb"
    if done(clean, cfg):
        return clean
    keep = []
    for l in _first_model(src.read_text().splitlines()):
        if l.startswith("ATOM  ") and _altloc_ok(l):
            if cfg.protein_chains and l[21] not in cfg.protein_chains:
                continue
            keep.append(_no_altloc(l))
    if not keep:
        raise RuntimeError("No protein ATOM records found in the input structure")
    raw.write_text("\n".join(keep) + "\nEND\n")

    if cfg.fix_protein:
        try:
            from openmm.app import PDBFile
            from pdbfixer import PDBFixer
            fx = PDBFixer(filename=str(raw))
            fx.findMissingResidues()
            fx.missingResidues = {}           # do not model missing loops/termini
            fx.findNonstandardResidues()
            fx.replaceNonstandardResidues()
            fx.removeHeterogens(keepWater=False)
            fx.findMissingAtoms()
            fx.addMissingAtoms()
            with open(clean, "w") as fh:
                PDBFile.writeFile(fx.topology, fx.positions, fh, keepIds=True)
            print("    PDBFixer: added missing heavy atoms, replaced non-standard residues")
            return clean
        except ImportError:
            print("    PDBFixer not installed - using the raw protein (conda install pdbfixer)")
    shutil.copy(raw, clean)
    return clean


def _extract_ligand_pdb(cfg: Config, src: Path, out: Path):
    lines = [l for l in _first_model(src.read_text().splitlines())
             if l.startswith(("HETATM", "ATOM  ")) and l[17:20].strip() == cfg.ligand_resname
             and _altloc_ok(l) and (cfg.ligand_chain is None or l[21] == cfg.ligand_chain)]
    if not lines:
        raise RuntimeError(f"Ligand '{cfg.ligand_resname}' not found in {src}")
    first = (lines[0][21], lines[0][22:27])          # first copy only (chain, resSeq)
    lines = [l for l in lines if (l[21], l[22:27]) == first]
    lines = [_no_altloc(l)[:17] + f"{LIG:>3}" + l[20:] for l in lines]
    out.write_text("\n".join(lines) + "\nEND\n")
    print(f"    extracted {len(lines)} atoms of {cfg.ligand_resname} "
          f"(chain {first[0]}, residue {first[1].strip()})")


def _split_section(text, section):
    """Split an .itp into (lines inside [ section ], all other lines)."""
    inside, other, cur = [], [], None
    for line in text.splitlines():
        m = re.match(r"\s*\[\s*(\w+)\s*\]", line)
        if m:
            cur = m.group(1).lower()
        (inside if cur == section else other).append(line)
    return inside, other


def process_ligand_itp(itp_text: str):
    """Return (atomtypes_itp, molecule_itp, posre_itp, molname) from an acpype itp."""
    atomtypes, rest = _split_section(itp_text, "atomtypes")
    out, cur, molname, heavy = [], None, None, []
    for line in rest:
        m = re.match(r"\s*\[\s*(\w+)\s*\]", line)
        if m:
            cur = m.group(1).lower()
            out.append(line)
            continue
        body = line.split(";")[0].split()
        if cur == "moleculetype" and body and molname is None:
            molname = body[0]
        if cur == "atoms" and len(body) >= 8:
            body[3] = LIG                                   # residue name
            if float(body[7]) > 1.5:                        # heavy atom
                heavy.append(int(body[0]))
            comment = line.split(";", 1)[1] if ";" in line else ""
            line = ("{:>6} {:>10} {:>6} {:>6} {:>6} {:>6} {:>12} {:>10}".format(*body[:8])
                    + ("".join(f" {x}" for x in body[8:]))
                    + (f" ;{comment}" if comment else ""))
        out.append(line)
    posre = ["; position restraints on ligand heavy atoms",
             "[ position_restraints ]", ";  ai  funct  fcx  fcy  fcz"]
    posre += [f"{i:6d}     1  1000  1000  1000" for i in heavy]
    return "\n".join(atomtypes) + "\n", "\n".join(out) + "\n", "\n".join(posre) + "\n", molname


def prepare_ligand(cfg: Config, P: Paths, R: Runner, src: Path) -> str:
    banner("Ligand parameters (GAFF2 / AM1-BCC via acpype)")
    itp_out = P.sys / f"{LIG}.itp"
    molname_file = P.lig / "molname.txt"
    if itp_out.exists() and molname_file.exists() and not cfg.overwrite:
        print("    already done - skipping")
        return molname_file.read_text().strip()

    mol2 = P.lig / "ligand.mol2"
    if cfg.ligand_file:
        lf = Path(cfg.ligand_file).expanduser()
        ext = lf.suffix.lower().lstrip(".")
        shutil.copy(lf, P.lig / f"ligand_input.{ext}")
        R.run(["obabel", f"ligand_input.{ext}", "-O", "ligand.mol2"], "01_ligand", cwd=P.lig)
        print("    using the supplied ligand file as-is (it must contain hydrogens)")
    else:
        _extract_ligand_pdb(cfg, src, P.lig / "ligand_raw.pdb")
        R.run(["obabel", "-ipdb", "ligand_raw.pdb", "-omol2", "-O", "ligand.mol2", "-p", str(cfg.ph)],
              "01_ligand", cwd=P.lig)
        print(f"    protonated at pH {cfg.ph} with Open Babel - CHECK ligand.mol2 (bond orders are "
              "guessed from PDB geometry; supply an SDF via ligand_file for drug-like ligands)")
    if not mol2.exists():
        raise RuntimeError("Open Babel did not produce ligand.mol2")

    cmd = ["acpype", "-i", "ligand.mol2", "-b", LIG, "-c", cfg.charge_method,
           "-a", cfg.ligand_ff, "-o", "gmx"]
    if cfg.ligand_charge is not None:
        cmd += ["-n", str(cfg.ligand_charge)]
    R.run(cmd, "01_ligand", cwd=P.lig)

    adir = P.lig / f"{LIG}.acpype"
    itp = adir / f"{LIG}_GMX.itp"
    gro = adir / f"{LIG}_GMX.gro"
    if not itp.exists():
        raise RuntimeError(f"acpype failed - see {P.logs / '01_ligand.log'}")

    types, mol, posre, molname = process_ligand_itp(itp.read_text())
    (P.sys / f"{LIG}_atomtypes.itp").write_text(types)
    itp_out.write_text(mol)
    (P.sys / f"posre_{LIG}.itp").write_text(posre)
    # ligand coordinates with residue name LIG
    g = gro.read_text().splitlines()
    n = int(g[1])
    atoms = [l[:5] + f"{LIG:<5}" + l[10:] for l in g[2:2 + n]]
    (P.sys / f"{LIG}.gro").write_text("\n".join([g[0], g[1], *atoms, g[2 + n]]) + "\n")
    molname_file.write_text(molname)
    print(f"    ligand topology ready (moleculetype '{molname}', {n} atoms)")
    return molname


def merge_gro(protein_gro: Path, ligand_gro: Path, out: Path):
    p = protein_gro.read_text().splitlines()
    l = ligand_gro.read_text().splitlines()
    pn, ln = int(p[1]), int(l[1])
    atoms = p[2:2 + pn] + l[2:2 + ln]
    out.write_text("\n".join(["Protein-ligand complex", f"{len(atoms):5d}", *atoms, p[2 + pn]]) + "\n")


def add_ligand_to_top(top: Path, molname: str):
    text = top.read_text()
    if f'#include "{LIG}.itp"' in text:
        return
    lines = text.splitlines()
    i = next(i for i, l in enumerate(lines) if re.search(r'#include\s+".*forcefield\.itp"', l))
    lines[i + 1:i + 1] = ["", "; Ligand atom types", f'#include "{LIG}_atomtypes.itp"']
    j = next((i for i, l in enumerate(lines) if "Include water topology" in l), None)
    if j is None:
        j = next(i for i, l in enumerate(lines) if re.match(r"\s*\[\s*system\s*\]", l))
    lines[j:j] = ["; Include ligand topology", f'#include "{LIG}.itp"',
                  f"#ifdef POSRES_{LIG}", f'#include "posre_{LIG}.itp"', "#endif", ""]
    lines.append(f"{molname:<20}1")
    top.write_text("\n".join(lines) + "\n")


def build_system(cfg: Config, P: Paths, R: Runner, protein_pdb: Path, molname: str | None):
    banner("Topology, box, solvent and ions")
    final = P.sys / "system.gro"
    if done(final, cfg):
        return final
    shutil.copy(protein_pdb, P.sys / "protein_clean.pdb")
    R.gmx(["pdb2gmx", "-f", "protein_clean.pdb", "-o", "protein.gro", "-p", "topol.top",
           "-i", "posre.itp", "-ff", cfg.forcefield, "-water", cfg.water, "-ignh"],
          "03_pdb2gmx", cwd=P.sys)
    if molname:
        merge_gro(P.sys / "protein.gro", P.sys / f"{LIG}.gro", P.sys / "complex.gro")
        add_ligand_to_top(P.sys / "topol.top", molname)
    else:
        shutil.copy(P.sys / "protein.gro", P.sys / "complex.gro")
    R.gmx(["editconf", "-f", "complex.gro", "-o", "boxed.gro", "-c",
           "-d", cfg.box_distance, "-bt", cfg.box_type], "03_box", cwd=P.sys)
    R.gmx(["solvate", "-cp", "boxed.gro", "-cs", "spc216.gro", "-o", "solvated.gro",
           "-p", "topol.top"], "03_solvate", cwd=P.sys)
    (P.sys / "ions.mdp").write_text(mdp_em(cfg))
    R.gmx(["grompp", "-f", "ions.mdp", "-c", "solvated.gro", "-p", "topol.top", "-o", "ions.tpr",
           "-po", "ions_out.mdp", "-maxwarn", max(cfg.maxwarn, 2)], "03_ions", cwd=P.sys)
    R.gmx(["genion", "-s", "ions.tpr", "-o", "system.gro", "-p", "topol.top",
           "-pname", cfg.positive_ion, "-nname", cfg.negative_ion,
           "-neutral", "-conc", cfg.salt_conc], "03_ions", cwd=P.sys, stdin="SOL\n")
    n = int((P.sys / "system.gro").read_text().splitlines()[1])
    print(f"    system built: {n:,} atoms")
    return final


# --------------------------------------------------------------------------- #
# MDP templates
# --------------------------------------------------------------------------- #
_COMMON = """cutoff-scheme       = Verlet
nstlist             = 10
pbc                 = xyz
coulombtype         = PME
rcoulomb            = 1.0
rvdw                = 1.0
fourierspacing      = 0.16
pme_order           = 4
DispCorr            = EnerPres
"""


def mdp_em(cfg):
    return f"""; energy minimisation
integrator          = steep
emtol               = 1000.0
emstep              = 0.01
nsteps              = {cfg.em_steps}
{_COMMON}"""


def _dyn(cfg, nsteps, nout, posres, pcoupl, continuation):
    define = f"-DPOSRES -DPOSRES_{LIG}" if posres else ""
    s = f"""define              = {define}
integrator          = md
dt                  = {cfg.dt}
nsteps              = {nsteps}
nstxout-compressed  = {nout}
nstenergy           = {nout}
nstlog              = {nout}
continuation        = {"yes" if continuation else "no"}
constraint_algorithm = lincs
constraints         = h-bonds
lincs_iter          = 1
lincs_order         = 4
{_COMMON}tcoupl              = V-rescale
tc-grps             = System
tau_t               = 0.1
ref_t               = {cfg.temperature}
ld_seed             = {cfg.seed}
"""
    if pcoupl:
        s += f"""pcoupl              = C-rescale
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = {cfg.pressure}
compressibility     = 4.5e-5
refcoord_scaling    = com
"""
    else:
        s += "pcoupl              = no\n"
    s += (f"gen_vel             = yes\ngen_temp            = {cfg.temperature}\ngen_seed            = {cfg.seed}\n"
          if not continuation else "gen_vel             = no\n")
    return s


def _steps(cfg, ps):
    return int(round(ps / cfg.dt))


# --------------------------------------------------------------------------- #
# Simulation stages
# --------------------------------------------------------------------------- #
def _gpu_variants(cfg, stage):
    if not cfg.use_gpu:
        return [[]]
    if stage == "em":
        return [["-nb", "gpu"], []]
    return [["-nb", "gpu", "-pme", "gpu", "-bonded", "gpu", "-update", "gpu"],
            ["-nb", "gpu", "-pme", "gpu", "-bonded", "gpu"],
            ["-nb", "gpu"],
            []]


def mdrun(cfg, R, workdir, deffnm, step, stage, cpi=None, extra=None):
    base = ["mdrun", "-deffnm", deffnm, "-cpt", cfg.checkpoint_min, *(extra or [])]
    if cfg.ntomp:
        base += ["-ntomp", cfg.ntomp]
    if cfg.use_gpu and "mpi" not in Path(cfg.gmx).name:
        base += ["-ntmpi", "1"]
    if cfg.use_gpu and cfg.gpu_id is not None:
        base += ["-gpu_id", cfg.gpu_id]
    if cpi:
        base += ["-cpi", cpi]
    for flags in _gpu_variants(cfg, stage):
        if R.gmx(base + flags, step, cwd=workdir, check=False) == 0:
            print(f"    mdrun OK  (offload: {' '.join(flags) if flags else 'CPU only'})")
            return flags
        print(f"    mdrun failed with [{' '.join(flags) or 'CPU'}] - retrying with less GPU offload")
    raise RuntimeError(f"mdrun failed for {deffnm}; see {R.p.logs / (step + '.log')} "
                       f"and {Path(workdir) / (deffnm + '.log')}")


def grompp(cfg, R, P, mdp_text, workdir: Path, name, step, coord, ref=None, cpt=None):
    (workdir / f"{name}.mdp").write_text(mdp_text)
    cmd = ["grompp", "-f", workdir / f"{name}.mdp", "-c", coord, "-p", "topol.top",
           "-o", workdir / f"{name}.tpr", "-po", workdir / f"{name}_out.mdp", "-maxwarn", cfg.maxwarn]
    if ref:
        cmd += ["-r", ref]
    if cpt:
        cmd += ["-t", cpt]
    R.gmx(cmd, step, cwd=P.sys)   # run in 03_system so relative #includes resolve


def performance(log: Path):
    if log.exists():
        m = re.search(r"Performance:\s+([\d.]+)", log.read_text(errors="replace"))
        return float(m.group(1)) if m else None
    return None


def run_simulations(cfg: Config, P: Paths, R: Runner) -> dict:
    used = run_equilibration(cfg, P, R)
    return run_production(cfg, P, R, used)


def run_equilibration(cfg: Config, P: Paths, R: Runner) -> dict:
    used = {}
    banner("Energy minimisation")
    if not done(P.em / "em.gro", cfg):
        grompp(cfg, R, P, mdp_em(cfg), P.em, "em", "04_em", P.sys / "system.gro")
        used["em"] = mdrun(cfg, R, P.em, "em", "04_em", "em")
        txt = (P.em / "em.log").read_text(errors="replace")
        for key in ("Potential Energy", "Maximum force"):
            m = re.search(rf"{key}\s*=\s*(\S+)", txt)
            if m:
                print(f"    {key}: {m.group(1)}")

    nout_eq = max(1, _steps(cfg, 1.0))  # every 1 ps during equilibration
    banner("NVT equilibration")
    if not done(P.nvt / "nvt.gro", cfg):
        grompp(cfg, R, P, _dyn(cfg, _steps(cfg, cfg.nvt_ps), nout_eq, True, False, False),
               P.nvt, "nvt", "05_nvt", P.em / "em.gro", ref=P.em / "em.gro")
        used["nvt"] = mdrun(cfg, R, P.nvt, "nvt", "05_nvt", "md")

    banner("NPT equilibration")
    if not done(P.npt / "npt.gro", cfg):
        grompp(cfg, R, P, _dyn(cfg, _steps(cfg, cfg.npt_ps), nout_eq, True, True, True),
               P.npt, "npt", "06_npt", P.nvt / "nvt.gro", ref=P.nvt / "nvt.gro", cpt=P.nvt / "nvt.cpt")
        used["npt"] = mdrun(cfg, R, P.npt, "npt", "06_npt", "md")
    return used


def run_production(cfg: Config, P: Paths, R: Runner, used: dict | None = None) -> dict:
    used = dict(used or {})
    banner(f"Production MD ({cfg.prod_ns} ns)")
    if not done(P.md / "md.gro", cfg):
        cpt = P.md / "md.cpt"
        resume = cpt.exists() and (P.md / "md.tpr").exists() and not cfg.overwrite
        if resume:
            print("    checkpoint found - resuming production run")
        else:
            grompp(cfg, R, P, _dyn(cfg, _steps(cfg, cfg.prod_ns * 1000),
                                   max(1, _steps(cfg, cfg.traj_interval_ps)), False, True, True),
                   P.md, "md", "07_md", P.npt / "npt.gro", cpt=P.npt / "npt.cpt")
        used["md"] = mdrun(cfg, R, P.md, "md", "07_md", "md", cpi=cpt if resume else None)
    perf = performance(P.md / "md.log")
    if perf:
        print(f"    production performance: {perf} ns/day")
    return {"mdrun_offload": used, "production_ns_per_day": perf}


# --------------------------------------------------------------------------- #
# Analysis
# --------------------------------------------------------------------------- #
def read_ndx_names(ndx: Path):
    return re.findall(r"^\s*\[\s*(.+?)\s*\]", ndx.read_text(), flags=re.M)


def make_index(cfg, P, R, tpr) -> list[str]:
    ndx = P.ana / "index.ndx"
    tmp = P.ana / "index_default.ndx"
    R.gmx(["make_ndx", "-f", tpr, "-o", tmp], "08_index", stdin="q\n")
    groups = read_ndx_names(tmp)
    cmds = []
    if cfg.has_ligand:
        if LIG not in groups:
            cmds += [f"r {LIG}", f"name {len(groups)} {LIG}"]
            groups.append(LIG)
        cmds += [f"{groups.index('Protein')} | {groups.index(LIG)}", f"name {len(groups)} Complex"]
    else:
        cmds += [f"{groups.index('Protein')}", f"name {len(groups)} Complex"]
    groups.append("Complex")
    R.gmx(["make_ndx", "-f", tpr, "-n", tmp, "-o", ndx], "08_index",
          stdin="\n".join(cmds) + "\nq\n")
    return read_ndx_names(ndx)


def read_xvg(path: Path):
    rows, legends, xl, yl, title = [], [], "", "", path.stem
    clean = lambda s: re.sub(r"\\[A-Za-z]", "", s)
    for line in path.read_text(errors="replace").splitlines():
        if line.startswith("#") or not line.strip():
            continue
        if line.startswith("@"):
            if m := re.search(r'xaxis\s+label\s+"(.*)"', line):
                xl = clean(m.group(1))
            elif m := re.search(r'yaxis\s+label\s+"(.*)"', line):
                yl = clean(m.group(1))
            elif m := re.search(r'^@\s+title\s+"(.*)"', line):
                title = clean(m.group(1))
            elif m := re.search(r's\d+\s+legend\s+"(.*)"', line):
                legends.append(clean(m.group(1)))
            continue
        try:
            rows.append([float(x) for x in line.split()])
        except ValueError:
            pass
    return rows, xl, yl, title, legends


def xvg_to_outputs(xvg: Path, csv_dir: Path, png_dir: Path):
    import matplotlib.pyplot as plt
    import numpy as np
    rows, xl, yl, title, legends = read_xvg(xvg)
    if not rows:
        return None
    width = min(len(r) for r in rows)
    a = np.array([r[:width] for r in rows])
    names = [xl or "x"] + (legends if len(legends) == width - 1 else
                           [yl or f"y{i}" for i in range(1, width)])
    np.savetxt(csv_dir / f"{xvg.stem}.csv", a, delimiter=",",
               header=",".join(n.replace(",", ";") for n in names), comments="")
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for i in range(1, width):
        ax.plot(a[:, 0], a[:, i], lw=0.9, label=names[i] if width > 2 else None)
    ax.set(xlabel=xl, ylabel=yl, title=title)
    if width > 2:
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(png_dir / f"{xvg.stem}.png", dpi=150)
    plt.close(fig)
    y = a[:, 1]
    half = y[len(y) // 2:]
    return {"label": yl, "mean": float(y.mean()), "std": float(y.std()),
            "min": float(y.min()), "max": float(y.max()), "final": float(y[-1]),
            "mean_second_half": float(half.mean()), "std_second_half": float(half.std())}


def analyse(cfg: Config, P: Paths, R: Runner) -> dict:
    banner("Analysis")
    xd, cd, pd = P.ana / "xvg", P.ana / "csv", P.ana / "plots"
    for d in (xd, cd, pd):
        d.mkdir(exist_ok=True)
    tpr, xtc = P.md / "md.tpr", P.md / "md.xtc"
    groups = make_index(cfg, P, R, tpr)
    ndx = P.ana / "index.ndx"
    g = lambda name: str(groups.index(name))
    lig = cfg.has_ligand
    fit = P.ana / "md_fit.xtc"

    if not done(fit, cfg):
        center = P.ana / "md_center.xtc"
        R.gmx(["trjconv", "-s", tpr, "-f", xtc, "-n", ndx, "-o", center, "-center",
               "-pbc", "mol", "-ur", "compact"], "08_trjconv",
              stdin=f"{g('Complex')}\n{g('System')}\n")
        R.gmx(["trjconv", "-s", tpr, "-f", center, "-n", ndx, "-o", fit, "-fit", "rot+trans"],
              "08_trjconv", stdin=f"{g('Backbone')}\n{g('System')}\n")
        center.unlink(missing_ok=True)

    common = ["-s", tpr, "-f", fit, "-n", ndx]

    def gmx_try(step, *variants):
        for args, stdin in variants:
            if R.gmx(args, step, stdin=stdin, check=False) == 0:
                return True
        print(f"    (skipped {step}: not supported by this GROMACS version - see logs)")
        return False

    gmx_try("08_rmsd", (["rms", *common, "-o", xd / "rmsd_backbone.xvg", "-tu", "ns"],
                        f"{g('Backbone')}\n{g('Backbone')}\n"))
    gmx_try("08_rmsf", (["rmsf", *common, "-o", xd / "rmsf_calpha.xvg", "-res"], f"{g('C-alpha')}\n"))
    gmx_try("08_gyrate",
            (["gyrate", *common, "-o", xd / "gyrate_protein.xvg"], f"{g('Protein')}\n"),
            (["gyrate", *common, "-o", xd / "gyrate_protein.xvg", "-sel", 'group "Protein"'], None))
    if lig:
        gmx_try("08_rmsd", (["rms", *common, "-o", xd / "rmsd_ligand.xvg", "-tu", "ns"],
                            f"{g('Backbone')}\n{g(LIG)}\n"))
        gmx_try("08_contacts", (["mindist", *common, "-od", xd / "mindist_protein_ligand.xvg",
                                 "-on", xd / "contacts_protein_ligand.xvg", "-d", "0.4", "-tu", "ns"],
                                f"{g('Protein')}\n{g(LIG)}\n"))
        gmx_try("08_hbond",
                (["hbond", *common, "-r", 'group "Protein"', "-t", f'group "{LIG}"',
                  "-num", xd / "hbonds_protein_ligand.xvg", "-tu", "ns"], None),
                (["hbond", *common, "-num", xd / "hbonds_protein_ligand.xvg", "-tu", "ns"],
                 f"{g('Protein')}\n{g(LIG)}\n"))

    energy_terms = [("em", P.em / "em.edr", "Potential"),
                    ("nvt", P.nvt / "nvt.edr", "Temperature"),
                    ("npt", P.npt / "npt.edr", "Pressure"), ("npt", P.npt / "npt.edr", "Density"),
                    ("md", P.md / "md.edr", "Potential"), ("md", P.md / "md.edr", "Temperature"),
                    ("md", P.md / "md.edr", "Pressure"), ("md", P.md / "md.edr", "Density")]
    for stage, edr, term in energy_terms:
        if edr.exists():
            R.gmx(["energy", "-f", edr, "-o", xd / f"energy_{stage}_{term.lower()}.xvg"],
                  "08_energy", stdin=f"{term}\n0\n", check=False)

    summary = {}
    for x in sorted(xd.glob("*.xvg")):
        s = xvg_to_outputs(x, cd, pd)
        if s:
            summary[x.stem] = s
    (P.ana / "summary.json").write_text(json.dumps(summary, indent=2))
    dashboard(P, summary)
    for k in ("rmsd_backbone", "rmsd_ligand", "contacts_protein_ligand", "hbonds_protein_ligand"):
        if k in summary:
            s = summary[k]
            print(f"    {k:<26} mean {s['mean']:.3f} ± {s['std']:.3f}  (2nd half {s['mean_second_half']:.3f})")
    return summary


def dashboard(P: Paths, summary: dict):
    import matplotlib.pyplot as plt
    import numpy as np
    panels = [("rmsd_backbone", "Backbone RMSD (nm)"), ("rmsd_ligand", "Ligand RMSD, fit on backbone (nm)"),
              ("rmsf_calpha", "C-alpha RMSF (nm)"), ("gyrate_protein", "Radius of gyration (nm)"),
              ("contacts_protein_ligand", "Protein-ligand contacts < 0.4 nm"),
              ("hbonds_protein_ligand", "Protein-ligand H-bonds")]
    panels = [p for p in panels if p[0] in summary]
    if not panels:
        return
    cols = 2
    rows = (len(panels) + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(11, 3.2 * rows), squeeze=False)
    for ax, (key, title) in zip(axes.flat, panels):
        a = np.loadtxt(P.ana / "csv" / f"{key}.csv", delimiter=",", skiprows=1, ndmin=2)
        xl = (P.ana / "csv" / f"{key}.csv").read_text().splitlines()[0].split(",")[0]
        ax.plot(a[:, 0], a[:, 1], lw=0.9)
        ax.set(title=title, xlabel=xl)
    for ax in list(axes.flat)[len(panels):]:
        ax.axis("off")
    fig.suptitle(P.root.name)
    fig.tight_layout()
    fig.savefig(P.ana / "dashboard.png", dpi=150)
    plt.close(fig)


# --------------------------------------------------------------------------- #
# Visualisation (ChimeraX + notebook)
# --------------------------------------------------------------------------- #
def export_viewer_files(cfg: Config, P: Paths, R: Runner):
    banner("Visualisation files")
    tpr, fit, ndx = P.md / "md.tpr", P.ana / "md_fit.xtc", P.ana / "index.ndx"
    groups = read_ndx_names(ndx)
    cpx = str(groups.index("Complex"))
    common = ["-s", tpr, "-f", fit, "-n", ndx]
    R.gmx(["trjconv", *common, "-o", P.viz / "complex_start.pdb", "-dump", "0"], "09_viz", stdin=cpx + "\n")
    R.gmx(["trjconv", *common, "-o", P.viz / "complex_final.pdb", "-dump", cfg.prod_ns * 1000],
          "09_viz", stdin=cpx + "\n")
    R.gmx(["trjconv", *common, "-o", P.viz / "complex_traj.xtc", "-skip", cfg.viewer_stride],
          "09_viz", stdin=cpx + "\n")
    write_chimerax_scripts(cfg, P)
    print(f"    ChimeraX scripts: {P.viz / 'view_trajectory.cxc'}\n"
          f"                      {P.viz / 'view_start_vs_final.cxc'}")
    export_chimera_files(cfg, P, R)


def export_chimera_files(cfg: Config, P: Paths, R: Runner):
    """Files for UCSF Chimera (the older program, not ChimeraX): a multi-model PDB
    trajectory for Tools > MD/Ensemble Analysis > MD Movie, plus .cmd style scripts."""
    tpr, fit, ndx = P.md / "md.tpr", P.ana / "md_fit.xtc", P.ana / "index.ndx"
    groups = read_ndx_names(ndx)
    cpx = str(groups.index("Complex"))
    raw = P.viz / "complex_traj_multimodel_raw.pdb"
    R.gmx(["trjconv", "-s", tpr, "-f", fit, "-n", ndx, "-o", raw,
           "-skip", cfg.chimera_pdb_stride], "09_viz", stdin=cpx + "\n")
    n = clean_multimodel_pdb(raw, P.viz / "complex_traj_multimodel.pdb")
    raw.unlink(missing_ok=True)
    print(f"    Chimera trajectory: {n} frames")
    write_chimera_scripts(cfg, P)
    print(f"    Chimera files:    {P.viz / 'complex_traj_multimodel.pdb'} (MD Movie, PDB format)\n"
          f"                      {P.viz / 'chimera_style.cmd'}, {P.viz / 'chimera_start_vs_final.cmd'}")


def clean_multimodel_pdb(src: Path, dst: Path) -> int:
    """Rewrite a GROMACS multi-frame PDB in the strict form Chimera's MD Movie expects:
    MODEL records numbered 1, 2, 3, ... in columns 11-14, only ATOM/HETATM/TER inside
    each model, ENDMDL after each, and a single END at the very end."""
    out, n, inside, frame = [], 0, False, []

    def flush():
        nonlocal n
        if frame:
            n += 1
            out.append(f"MODEL     {n:4d}")
            out.extend(frame)
            out.append("ENDMDL")

    for line in Path(src).read_text(errors="replace").splitlines():
        rec = line[:6]
        if rec.startswith("MODEL"):
            flush()
            frame, inside = [], True
        elif rec.startswith("ENDMDL"):
            flush()
            frame, inside = [], False
        elif rec in ("ATOM  ", "HETATM"):
            line = line.ljust(80)
            # fill a blank chain ID so every field has a value
            if line[21] == " ":
                line = line[:21] + "A" + line[22:]
            frame.append(line.rstrip())
        elif rec.startswith("TER"):
            frame.append("TER")
    flush()  # file without a final ENDMDL
    out.append("END")
    Path(dst).write_text("\n".join(out) + "\n")
    return n


def _chimera_style(model, lig):
    s = [f"~display {model}", f"ribbon {model}", f"ribcolor lightgray {model}"]
    if lig:
        s += [f"display {model}:{LIG}", f"represent bs {model}:{LIG}",
              f"color orange {model}:{LIG}", f"color byhet {model}:{LIG}",
              f"display {model}:{LIG} z<4.5 & protein & ~@H*",
              f"represent stick {model}:{LIG} z<4.5 & protein"]
    return s


def write_chimera_scripts(cfg: Config, P: Paths):
    lig = cfg.has_ligand
    head = ["# UCSF Chimera command script generated by md_pipeline.py",
            "background solid white", "set silhouette"]
    style = head + ["# Run AFTER loading the trajectory with MD Movie (it becomes model #0):",
                    "#   Tools > MD/Ensemble Analysis > MD Movie > format PDB > complex_traj_multimodel.pdb",
                    "# then File > Open > chimera_style.cmd"] + _chimera_style("#0", lig)
    if lig:
        style += [f"select #0:{LIG}", "findhbond selRestrict cross reveal true", "~select",
                  f"focus #0:{LIG}",
                  "# To update H-bonds every frame: MD Movie > Per-Frame > Define script... (Chimera commands)",
                  f"#   select #0:{LIG}; findhbond selRestrict cross reveal true; ~select"]
    else:
        style += ["focus"]
    (P.viz / "chimera_style.cmd").write_text("\n".join(style) + "\n")

    cmp_ = head + ["# Open from the 09_visualization folder (or run: cd <that folder>)",
                   "open #0 complex_start.pdb", "open #1 complex_final.pdb"]
    cmp_ += _chimera_style("#0", lig) + _chimera_style("#1", lig)
    cmp_ += ["ribcolor cornflower blue #0", "ribcolor salmon #1"]
    if lig:
        cmp_ += [f"color cornflower blue #0:{LIG}", f"color byhet #0:{LIG}",
                 f"focus #0:{LIG}", f"rmsd #0:{LIG}&~@H* #1:{LIG}&~@H*"]
    else:
        cmp_ += ["focus"]
    (P.viz / "chimera_start_vs_final.cmd").write_text("\n".join(cmp_) + "\n")


def _style(model, lig):
    s = [f"hide {model} atoms", f"cartoon {model}", f"color {model} & protein lightgray target c"]
    if lig:
        s += [f"show {model} & :{LIG} atoms", f"style {model} & :{LIG} ball",
              f"color {model} & :{LIG} & C orange", f"color {model} & :{LIG} byhetero",
              f"show {model} & (:{LIG} :<4.5) & protein & ~HC atoms",
              f"style {model} & (:{LIG} :<4.5) & protein stick"]
    return s


def write_chimerax_scripts(cfg: Config, P: Paths):
    lig = cfg.has_ligand
    head = ["# Generated by md_pipeline.py. Paths are relative to this folder:",
            "# run with `chimerax <this file>` from this folder, or File > Open in ChimeraX.",
            "set bgColor white", "lighting soft", "graphics silhouettes true"]
    traj = head + ["open complex_start.pdb", "open complex_traj.xtc structureModel #1"] + _style("#1", lig)
    if lig:
        traj += [f"hbonds #1 & :{LIG} restrict cross reveal true name ligand_hbonds",
                 f"view #1 & :{LIG} pad 0.8",
                 "# recompute H-bonds on every frame while playing (slower):",
                 f'# perframe "hbonds #1 & :{LIG} restrict cross reveal true name ligand_hbonds"']
    else:
        traj += ["view"]
    traj += ["coordset slider #1"]
    (P.viz / "view_trajectory.cxc").write_text("\n".join(traj) + "\n")

    cmp_ = head + ["open complex_start.pdb", "open complex_final.pdb"] + _style("#1", lig) + _style("#2", lig)
    cmp_ += ["color #1 & protein lightblue target c", "color #2 & protein salmon target c"]
    if lig:
        cmp_ += [f"color #1 & :{LIG} & C cornflowerblue", f"color #2 & :{LIG} & C orange",
                 f"view :{LIG} pad 0.8",
                 f"rmsd #1 & :{LIG} & ~H to #2 & :{LIG} & ~H"]
    else:
        cmp_ += ["view"]
    cmp_ += ["2dlabels text 'start (blue)  vs  final (salmon/orange)' xpos 0.03 ypos 0.95 size 18"]
    (P.viz / "view_start_vs_final.cxc").write_text("\n".join(cmp_) + "\n")


def find_chimerax(cfg: Config | None = None) -> str | None:
    if cfg and cfg.chimerax_path:
        return cfg.chimerax_path
    for name in ("chimerax", "ChimeraX", "chimerax-daily"):
        if p := shutil.which(name):
            return p
    patterns = ["/Applications/ChimeraX*.app/Contents/MacOS/ChimeraX",
                "/usr/bin/chimerax*", "/opt/UCSF/ChimeraX*/bin/ChimeraX",
                "/usr/lib/ucsf-chimerax*/bin/ChimeraX",
                r"C:\Program Files\ChimeraX*\bin\ChimeraX.exe",
                "/mnt/c/Program Files/ChimeraX*/bin/ChimeraX.exe"]   # Windows ChimeraX from WSL
    for pat in patterns:
        hits = sorted(glob.glob(pat), reverse=True)
        if hits:
            return hits[0]
    return None


def open_in_chimerax(cfg: Config, view: str = "trajectory"):
    """Launch ChimeraX on your desktop. view = 'trajectory' or 'start_vs_final'."""
    P = Paths(cfg)
    cxc = P.viz / f"view_{view}.cxc"
    if not cxc.exists():
        raise FileNotFoundError(f"{cxc} not found - run the pipeline first")
    exe = find_chimerax(cfg)
    if not exe:
        print("ChimeraX not found. Set Config(chimerax_path='/path/to/ChimeraX') or open this file "
              f"manually in ChimeraX:\n  {cxc}")
        return
    arg = str(cxc)
    if exe.lower().endswith(".exe") and shutil.which("wslpath"):
        arg = subprocess.check_output(["wslpath", "-w", arg], text=True).strip()
    kw = {"start_new_session": True} if os.name == "posix" else {}
    subprocess.Popen([exe, arg], cwd=P.viz, stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, **kw)
    print(f"Opened {cxc.name} in ChimeraX ({exe})")


def package_for_download(cfg: Config, include_full_trajectory: bool = False) -> Path:
    """Zip the visualisation + analysis folders (and optionally the full trajectory) for
    download, e.g. from Colab to open the ChimeraX scripts on your own computer."""
    import zipfile
    P = Paths(cfg)
    out = P.root / f"{cfg.name}_results.zip"
    keep = [P.viz, P.ana / "plots", P.ana / "csv", P.ana / "xvg"]
    extra = [P.ana / "summary.json", P.ana / "dashboard.png", P.ana / "index.ndx",
             P.root / "config.json", P.root / "run_info.json", P.root / "README.txt",
             P.sys / "topol.top", P.md / "md.tpr"]
    if include_full_trajectory:
        extra.append(P.ana / "md_fit.xtc")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for d in keep:
            for f in d.rglob("*") if d.exists() else []:
                if f.is_file():
                    z.write(f, f.relative_to(P.root))
        for f in extra:
            if f.exists():
                z.write(f, f.relative_to(P.root))
    print(f"Packed {out} ({out.stat().st_size / 1e6:.1f} MB). Unzip it and open "
          "09_visualization/view_trajectory.cxc in ChimeraX.")
    return out


def show_in_notebook(cfg: Config, which: str = "final", width=700, height=500):
    """Quick inline 3D view in Colab/Jupyter with py3Dmol (which = 'start' or 'final')."""
    import py3Dmol
    pdb = Paths(cfg).viz / f"complex_{which}.pdb"
    v = py3Dmol.view(width=width, height=height)
    v.addModel(pdb.read_text(), "pdb")
    v.setStyle({"cartoon": {"color": "spectrum"}})
    if cfg.has_ligand:
        v.setStyle({"resn": LIG}, {"stick": {"colorscheme": "orangeCarbon"}})
        v.addStyle({"within": {"distance": 4.5, "sel": {"resn": LIG}}, "not": {"resn": LIG}},
                   {"stick": {"radius": 0.15}})
        v.zoomTo({"resn": LIG})
    else:
        v.zoomTo()
    return v.show()


# --------------------------------------------------------------------------- #
# Driver
# --------------------------------------------------------------------------- #
README = """Results layout
==============
00_input/          input structure
01_ligand/         ligand extraction, protonation, acpype output (GAFF2/AM1-BCC)
02_protein/        cleaned protein
03_system/         topol.top, *.itp, system.gro  (full solvated, neutralised system)
04_em/ 05_nvt/ 06_npt/   minimisation and equilibration (.tpr .gro .edr .log .cpt)
07_production/     md.tpr, md.xtc (raw trajectory), md.edr, md.log, md.cpt
08_analysis/       md_fit.xtc  - PBC-corrected, fitted full trajectory (use with md.tpr)
                   index.ndx   - groups incl. LIG and Complex (protein+ligand)
                   xvg/ csv/ plots/, dashboard.png, summary.json
09_visualization/  complex_start.pdb, complex_final.pdb, complex_traj.xtc (protein+ligand only)
                   view_trajectory.cxc, view_start_vs_final.cxc (ChimeraX)
                   complex_traj_multimodel.pdb, chimera_style.cmd,
                   chimera_start_vs_final.cmd (UCSF Chimera, via MD Movie)
logs/              every command and its output
config.json        exact settings used; run_info.json - versions, GPU, performance

Python follow-up, e.g. with MDAnalysis:
  import MDAnalysis as mda
  u = mda.Universe("09_visualization/complex_start.pdb", "09_visualization/complex_traj.xtc")
"""


def prepare_equilibrated(cfg: Config):
    """Build the system and run EM, NVT and NPT (finished steps are skipped).
    Returns (Paths, Runner, info). Used by run_pipeline and by md_advanced."""
    P = Paths(cfg)
    P.make()
    R = Runner(cfg, P)
    print(f"Results folder: {P.root}")
    info = check_environment(cfg)
    (P.root / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
    (P.root / "README.txt").write_text(README)

    src = get_structure(cfg, P)
    protein = prepare_protein(cfg, P, src)
    molname = prepare_ligand(cfg, P, R, src) if cfg.has_ligand else None
    build_system(cfg, P, R, protein, molname)
    info["mdrun_offload_equil"] = run_equilibration(cfg, P, R)
    return P, R, info


def run_pipeline(cfg: Config) -> dict:
    t0 = time.time()
    P, R, info = prepare_equilibrated(cfg)
    info.update(run_production(cfg, P, R))
    summary = analyse(cfg, P, R)
    export_viewer_files(cfg, P, R)

    info["wall_time_min"] = round((time.time() - t0) / 60, 1)
    (P.root / "run_info.json").write_text(json.dumps(info, indent=2))
    banner("Done")
    print(f"All results in: {P.root}\nDashboard:      {P.ana / 'dashboard.png'}")
    if "ipykernel" in sys.modules and (P.ana / "dashboard.png").exists():
        try:
            from IPython.display import Image, display
            display(Image(str(P.ana / "dashboard.png")))
        except ImportError:
            pass
    if cfg.launch_chimerax:
        open_in_chimerax(cfg)
    return {"paths": P, "summary": summary, "info": info}


def _cli():
    ap = argparse.ArgumentParser(description="Protein-ligand MD with GROMACS on a local GPU")
    ap.add_argument("--config", help="JSON file with Config fields (CLI flags override it)")
    ap.add_argument("--name")
    ap.add_argument("--pdb-id")
    ap.add_argument("--pdb-file")
    ap.add_argument("--ligand-resname")
    ap.add_argument("--ligand-file")
    ap.add_argument("--ligand-charge", type=int)
    ap.add_argument("--apo", action="store_true", help="no ligand (protein only)")
    ap.add_argument("--prod-ns", type=float)
    ap.add_argument("--temperature", type=float)
    ap.add_argument("--output-root")
    ap.add_argument("--gmx")
    ap.add_argument("--gpu-id")
    ap.add_argument("--ntomp", type=int)
    ap.add_argument("--cpu", action="store_true", help="do not use the GPU")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--chimerax", action="store_true", help="open ChimeraX at the end")
    a = ap.parse_args()

    cfg = Config.from_json(a.config) if a.config else Config()
    for k in ("name", "pdb_id", "pdb_file", "ligand_resname", "ligand_file", "ligand_charge",
              "prod_ns", "temperature", "output_root", "gmx", "gpu_id", "ntomp"):
        v = getattr(a, k)
        if v is not None:
            setattr(cfg, k, v)
    if a.pdb_file and not a.pdb_id:
        cfg.pdb_id = None
    if a.ligand_file and not a.ligand_resname:
        cfg.ligand_resname = None
    if a.apo:
        cfg.ligand_resname = cfg.ligand_file = None
    cfg.use_gpu = not a.cpu
    cfg.overwrite = a.overwrite
    cfg.launch_chimerax = a.chimerax
    run_pipeline(cfg)


if __name__ == "__main__":
    _cli()

In [ ]:
%%writefile /content/md_advanced.py
"""
md_advanced.py  -  Levels 2-5 of the MD learning path, built on md_pipeline.py
==============================================================================

Every function reuses the system that md_pipeline builds (same Config), and writes
into a sub-folder of that system's results folder:

  Level 2  run_replicas / compare_replicas / pca_replicas / run_apo / compare_rmsf
           -> <name>/L2_replicas/rep1..N, <name>/L2_replicas/comparison, <name>/L2_apo
  Level 3  run_mmpbsa                 -> <name>/L3_mmpbsa          (gmx_MMPBSA, MM/GBSA)
           run_hydration_fe / analyse_hydration_fe
                                       -> <name>/L3_hydration       (alchemical, MBAR)
  Level 4  smd_config / run_smd / analyse_smd
                                       -> <name>_bigbox/L4_smd      (steered MD, Jarzynski)
  Level 5  run_umbrella / run_wham     -> <name>_bigbox/L5_umbrella (umbrella sampling, WHAM)
           dipeptide_config / build_dipeptide / run_dipeptide_metad / analyse_dipeptide_metad
                                       -> ala_dipeptide/L5_metad    (PLUMED metadynamics)
           run_ligand_metad / analyse_ligand_metad
                                       -> <name>/L5_metad_ligand    (optional)

All long runs checkpoint and resume: re-run the same cell after a Colab disconnect.
"""
from __future__ import annotations

import json
import math
import re
import shutil
import subprocess
from dataclasses import replace
from pathlib import Path

import md_pipeline as mp
from md_pipeline import LIG, Config, Paths, Runner, banner

KB_KJ = 0.0083144626      # kJ/mol/K
KB_KCAL = 0.0019872043    # kcal/mol/K
KJ_PER_KCAL = 4.184


# =========================================================================== #
# Shared helpers
# =========================================================================== #
def stage_dir(cfg: Config, name: str) -> Path:
    d = Paths(cfg).root / name
    d.mkdir(parents=True, exist_ok=True)
    return d


def runner_for(cfg: Config) -> tuple[Paths, Runner]:
    P = Paths(cfg)
    P.make()
    return P, Runner(cfg, P)


def read_gro(path: Path):
    """Return (atoms, box) from a .gro file. atoms: list of dicts with
    resnr, resname, name, index (1-based), xyz (nm)."""
    lines = Path(path).read_text().splitlines()
    n = int(lines[1])
    atoms = []
    for i, l in enumerate(lines[2:2 + n]):
        atoms.append({"resnr": int(l[0:5]), "resname": l[5:10].strip(), "name": l[10:15].strip(),
                      "index": i + 1,
                      "xyz": (float(l[20:28]), float(l[28:36]), float(l[36:44]))})
    box = [float(x) for x in lines[2 + n].split()]
    return atoms, box


def box_matrix(box):
    """3x3 box (rows = box vectors) from a .gro box line."""
    if len(box) == 3:
        return [[box[0], 0, 0], [0, box[1], 0], [0, 0, box[2]]]
    v1x, v2y, v3z, v1y, v1z, v2x, v2z, v3x, v3y = box[:9]
    return [[v1x, v1y, v1z], [v2x, v2y, v2z], [v3x, v3y, v3z]]


def min_image(vec, box):
    """Shortest periodic image of a difference vector (works for triclinic boxes)."""
    M = box_matrix(box)
    best, bestd = vec, float("inf")
    for i in (-1, 0, 1):
        for j in (-1, 0, 1):
            for k in (-1, 0, 1):
                v = [vec[a] + i * M[0][a] + j * M[1][a] + k * M[2][a] for a in range(3)]
                d = sum(x * x for x in v)
                if d < bestd:
                    best, bestd = v, d
    return best


def centre(xyzs):
    n = len(xyzs)
    return [sum(p[a] for p in xyzs) / n for a in range(3)]


def centre_pbc(xyzs, box):
    """Geometric centre of a group that may be split across the periodic box.
    Each point is unwrapped next to the PREVIOUS one (consecutive atoms of a chain are
    always close), so groups larger than half the box are handled correctly."""
    pts = [list(xyzs[0])]
    for p in xyzs[1:]:
        prev = pts[-1]
        d = min_image([p[a] - prev[a] for a in range(3)], box)
        pts.append([prev[a] + d[a] for a in range(3)])
    return centre(pts)


def protein_atoms(atoms):
    """Atoms before the first ligand / water / ion residue."""
    stop = {LIG, "SOL", "HOH", "WAT", "NA", "CL", "K", "MG", "CA2", "ZN"}
    out = []
    for a in atoms:
        if a["resname"] in stop:
            break
        out.append(a)
    return out


def write_ndx(path: Path, groups: dict):
    with open(path, "w") as fh:
        for name, idx in groups.items():
            fh.write(f"[ {name} ]\n")
            for i in range(0, len(idx), 15):
                fh.write(" ".join(f"{x:5d}" for x in idx[i:i + 15]) + "\n")
            fh.write("\n")


def read_ndx(path: Path) -> dict:
    groups, cur = {}, None
    for line in Path(path).read_text().splitlines():
        m = re.match(r"\s*\[\s*(.+?)\s*\]", line)
        if m:
            cur = m.group(1)
            groups[cur] = []
        elif cur and line.strip():
            groups[cur] += [int(x) for x in line.split()]
    return groups


def xvg_array(path: Path):
    import numpy as np
    rows, *_ = mp.read_xvg(Path(path))
    w = min(len(r) for r in rows)
    return np.array([r[:w] for r in rows])


def set_define(mdp: str, define: str) -> str:
    return re.sub(r"^define\s*=.*$", f"define              = {define}", mdp, flags=re.M)


def set_mdp(mdp: str, key: str, value) -> str:
    """Replace `key = ...` if present, else append it."""
    pat = rf"^{re.escape(key)}\s*=.*$"
    line = f"{key:<20}= {value}"
    if re.search(pat, mdp, flags=re.M):
        return re.sub(pat, line, mdp, flags=re.M)
    return mdp.rstrip("\n") + "\n" + line + "\n"


def run_md(cfg: Config, P: Paths, R: Runner, workdir: Path, name: str, mdp: str, coord: Path,
           cpt: Path | None = None, ref: Path | None = None, ndx: Path | None = None,
           extra: list | None = None, plumed: str | None = None, step: str | None = None,
           stage: str = "md") -> bool:
    """grompp + mdrun with resume. Finished = <name>.gro exists. Returns True if it ran."""
    workdir.mkdir(parents=True, exist_ok=True)
    step = step or f"{workdir.parent.name}_{workdir.name}_{name}"
    if (workdir / f"{name}.gro").exists() and not cfg.overwrite:
        print(f"    {workdir.name}/{name}: already done - skipping")
        return False
    tpr, cp = workdir / f"{name}.tpr", workdir / f"{name}.cpt"
    resume = tpr.exists() and cp.exists() and not cfg.overwrite
    if not resume:
        (workdir / f"{name}.mdp").write_text(mdp)
        cmd = ["grompp", "-f", workdir / f"{name}.mdp", "-c", coord, "-p", "topol.top", "-o", tpr,
               "-po", workdir / f"{name}_out.mdp", "-maxwarn", cfg.maxwarn]
        if ref:
            cmd += ["-r", ref]
        if cpt:
            cmd += ["-t", cpt]
        if ndx:
            cmd += ["-n", ndx]
        R.gmx(cmd, step, cwd=P.sys)
    else:
        print(f"    {workdir.name}/{name}: checkpoint found - resuming")
    extra = list(extra or [])
    if plumed is not None:
        (workdir / "plumed.dat").write_text(("RESTART\n" if resume else "") + plumed)
        extra += ["-plumed", "plumed.dat"]
        if not cfg.use_gpu:                      # PLUMED cannot run with >1 thread-MPI rank
            extra += ["-ntmpi", "1"]
    mp.mdrun(cfg, R, workdir, name, step, stage, cpi=cp if resume else None, extra=extra)
    return True


def reuse_inputs(src: Config, dst: Config):
    """Copy downloaded structure, ligand parameters and cleaned protein from one system
    folder to another, so a rebuilt system (e.g. a bigger box) skips those steps."""
    a, b = Paths(src), Paths(dst)
    b.make()
    for d in ("inp", "lig", "prot"):
        s, t = getattr(a, d), getattr(b, d)
        if s.exists() and not any(t.iterdir()):
            shutil.copytree(s, t, dirs_exist_ok=True)
    for f in (f"{LIG}.itp", f"{LIG}_atomtypes.itp", f"posre_{LIG}.itp", f"{LIG}.gro"):
        if (a.sys / f).exists() and not (b.sys / f).exists():
            shutil.copy(a.sys / f, b.sys / f)


def plot_save(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    try:
        from IPython.display import display
        display(fig)
    except ImportError:
        pass
    import matplotlib.pyplot as plt
    plt.close(fig)


# =========================================================================== #
# Level 2 - replicas, statistics, PCA, apo vs holo
# =========================================================================== #
def replica_config(base: Config, i: int, prod_ns: float | None = None) -> Config:
    return replace(base, name=f"{base.name}/L2_replicas/rep{i}", seed=1000 + i,
                   prod_ns=prod_ns if prod_ns is not None else base.prod_ns, launch_chimerax=False)


def run_replicas(base: Config, n: int = 3, prod_ns: float | None = None) -> list[Config]:
    """n independent replicas: same minimised structure, new random velocities
    (seed 1001, 1002, ...), then NVT, NPT, production and the standard analysis."""
    banner(f"Level 2: {n} replicas")
    P0, _, _ = mp.prepare_equilibrated(base)           # makes sure the base system exists
    cfgs = []
    for i in range(1, n + 1):
        rc = replica_config(base, i, prod_ns)
        Pi = Paths(rc)
        Pi.make()
        if not (Pi.em / "em.gro").exists():
            for d in ("inp", "lig", "prot", "sys", "em"):
                shutil.copytree(getattr(P0, d), getattr(Pi, d), dirs_exist_ok=True)
        banner(f"Replica {i} of {n} (seed {rc.seed})")
        mp.run_pipeline(rc)
        cfgs.append(rc)
    return cfgs


def block_average(x, max_blocks: int = 64):
    """Standard error of the mean vs block length (Flyvbjerg-Petersen style).
    Returns (block_lengths, sem). The plateau is the honest error bar."""
    import numpy as np
    x = np.asarray(x, float)
    sizes, sems = [], []
    b = 1
    while len(x) // b >= 4:
        nb = len(x) // b
        means = x[:nb * b].reshape(nb, b).mean(1)
        sizes.append(b)
        sems.append(means.std(ddof=1) / math.sqrt(nb))
        b *= 2
        if len(sizes) > max_blocks:
            break
    return np.array(sizes), np.array(sems)


def autocorr_time(x):
    """Integrated autocorrelation time (in samples), summed until the ACF first drops below 0."""
    import numpy as np
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    if n < 4 or not x.any():
        return 1.0
    f = np.fft.rfft(x, 2 * n)
    acf = np.fft.irfft(f * np.conj(f))[:n]
    acf /= acf[0]
    tau = 1.0
    for k in range(1, n):
        if acf[k] < 0:
            break
        tau += 2 * acf[k]
    return float(tau)


METRICS = [("rmsd_backbone", "Backbone RMSD (nm)"), ("rmsd_ligand", "Ligand RMSD (nm)"),
           ("contacts_protein_ligand", "Protein-ligand contacts"),
           ("hbonds_protein_ligand", "Protein-ligand H-bonds"), ("gyrate_protein", "Radius of gyration (nm)")]


def compare_replicas(cfgs: list[Config], out_cfg: Config | None = None):
    """Mean +/- SEM across replicas (second half of each run), overlay plots,
    block averaging and autocorrelation of the ligand RMSD."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    out_cfg = out_cfg or cfgs[0]
    base_name = out_cfg.name.split("/L2_replicas")[0]
    out = Paths(replace(out_cfg, name=base_name)).root / "L2_replicas" / "comparison"
    out.mkdir(parents=True, exist_ok=True)
    labels = [c.name.split("/")[-1] for c in cfgs]

    rows = []
    for key, label in METRICS:
        vals = []
        for c in cfgs:
            s = json.loads((Paths(c).ana / "summary.json").read_text())
            if key in s:
                vals.append(s[key]["mean_second_half"])
        if len(vals) >= 2:
            v = np.array(vals)
            rows.append({"quantity": label, **{l: round(x, 4) for l, x in zip(labels, vals)},
                         "mean": v.mean(), "sem": v.std(ddof=1) / math.sqrt(len(v)), "n": len(v)})
    table = pd.DataFrame(rows)
    table.to_csv(out / "replica_summary.csv", index=False)

    for key, label in METRICS:
        files = [(l, Paths(c).ana / "csv" / f"{key}.csv") for l, c in zip(labels, cfgs)]
        files = [(l, f) for l, f in files if f.exists()]
        if not files:
            continue
        fig, ax = plt.subplots(figsize=(7, 3.4))
        for l, f in files:
            a = np.loadtxt(f, delimiter=",", skiprows=1, ndmin=2)
            ax.plot(a[:, 0], a[:, 1], lw=0.8, label=l)
        ax.set(title=label, xlabel=f.read_text().splitlines()[0].split(",")[0], ylabel=label)
        ax.legend(fontsize=8)
        plot_save(fig, out / f"overlay_{key}.png")

    # block averaging + autocorrelation on ligand RMSD (or backbone RMSD for apo)
    key = "rmsd_ligand" if (Paths(cfgs[0]).ana / "csv" / "rmsd_ligand.csv").exists() else "rmsd_backbone"
    fig, ax = plt.subplots(figsize=(6, 3.4))
    acf_rows = []
    for l, c in zip(labels, cfgs):
        a = np.loadtxt(Paths(c).ana / "csv" / f"{key}.csv", delimiter=",", skiprows=1, ndmin=2)
        x = a[len(a) // 2:, 1]
        sizes, sems = block_average(x)
        dt = (a[1, 0] - a[0, 0]) if len(a) > 1 else 1
        ax.plot(sizes * dt, sems, "o-", ms=3, label=l)
        tau = autocorr_time(x)
        acf_rows.append({"replica": l, "tau_samples": round(tau, 1), "tau_time": round(tau * dt, 4),
                         "n_samples": len(x), "n_effective": round(len(x) / tau, 1)})
    ax.set(xscale="log", xlabel="Block length (same unit as time axis)", ylabel="SEM",
           title=f"Block averaging, {key} (2nd half)")
    ax.legend(fontsize=8)
    plot_save(fig, out / f"block_average_{key}.png")
    acf = pd.DataFrame(acf_rows)
    acf.to_csv(out / f"autocorrelation_{key}.csv", index=False)
    print(f"Saved to {out}")
    return table, acf


def _kabsch(X, ref):
    import numpy as np
    Xc, Rc = X - X.mean(0), ref - ref.mean(0)
    U, S, Vt = np.linalg.svd(Xc.T @ Rc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    Rm = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return Xc @ Rm.T + ref.mean(0)


def pca_replicas(cfgs: list[Config], selection: str = f"name CA and not resname {LIG}", n_pc: int = 5):
    """PCA of the combined C-alpha trajectories (all frames aligned to replica 1, frame 0).
    Uses the protein+ligand files in 09_visualization."""
    import warnings
    warnings.filterwarnings("ignore", module="MDAnalysis")
    import MDAnalysis as mda
    import numpy as np
    import matplotlib.pyplot as plt
    out = Paths(replace(cfgs[0], name=cfgs[0].name.split("/L2_replicas")[0])).root / "L2_replicas" / "comparison"
    out.mkdir(parents=True, exist_ok=True)
    blocks, labels, ref = [], [], None
    for c in cfgs:
        v = Paths(c).viz
        u = mda.Universe(str(v / "complex_start.pdb"), str(v / "complex_traj.xtc"))
        sel = u.select_atoms(selection)
        frames = []
        for _ in u.trajectory:
            X = sel.positions.astype(float)
            if ref is None:
                ref = X.copy()
            frames.append(_kabsch(X, ref))
        blocks.append(np.array(frames))
        labels.append(c.name.split("/")[-1])
    allX = np.concatenate([b.reshape(len(b), -1) for b in blocks])
    mean = allX.mean(0)
    _, S, Vt = np.linalg.svd(allX - mean, full_matrices=False)
    var = S ** 2 / (S ** 2).sum()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    start = 0
    for lab, b in zip(labels, blocks):
        proj = (b.reshape(len(b), -1) - mean) @ Vt[:2].T / 10      # Å -> nm
        a1.scatter(proj[:, 0], proj[:, 1], s=4, alpha=0.6, label=lab)
        np.savetxt(out / f"pca_projection_{lab}.csv", proj, delimiter=",", header="PC1_nm,PC2_nm", comments="")
        start += len(b)
    a1.set(xlabel=f"PC1 ({var[0]:.0%})", ylabel=f"PC2 ({var[1]:.0%})", title="Replicas in PC1/PC2 space")
    a1.legend(fontsize=8, markerscale=3)
    a2.bar(range(1, n_pc + 1), var[:n_pc] * 100)
    a2.set(xlabel="Principal component", ylabel="Variance explained (%)", title="Variance per PC")
    plot_save(fig, out / "pca.png")
    return var[:n_pc]


def run_apo(base: Config, prod_ns: float | None = None) -> Config:
    """Same protein without the ligand, for apo vs holo comparison."""
    apo = replace(base, name=f"{base.name}/L2_apo", ligand_resname=None, ligand_file=None,
                  prod_ns=prod_ns if prod_ns is not None else base.prod_ns, launch_chimerax=False)
    Pa, Pb = Paths(apo), Paths(base)
    Pa.make()
    for d in ("inp", "prot"):
        if not any(getattr(Pa, d).iterdir()):
            shutil.copytree(getattr(Pb, d), getattr(Pa, d), dirs_exist_ok=True)
    mp.run_pipeline(apo)
    return apo


def compare_rmsf(holo_cfgs: list[Config], apo_cfg: Config, highlight=(107, 115)):
    import numpy as np
    import matplotlib.pyplot as plt
    out = Paths(replace(holo_cfgs[0], name=holo_cfgs[0].name.split("/L2_replicas")[0])).root / "L2_replicas" / "comparison"
    out.mkdir(parents=True, exist_ok=True)
    holo = [np.loadtxt(Paths(c).ana / "csv" / "rmsf_calpha.csv", delimiter=",", skiprows=1, ndmin=2)
            for c in holo_cfgs]
    apo = np.loadtxt(Paths(apo_cfg).ana / "csv" / "rmsf_calpha.csv", delimiter=",", skiprows=1, ndmin=2)
    n = min(len(h) for h in holo)
    H = np.array([h[:n, 1] for h in holo])
    res = holo[0][:n, 0]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.fill_between(res, H.mean(0) - H.std(0), H.mean(0) + H.std(0), alpha=0.3, label="holo, mean ± SD")
    ax.plot(res, H.mean(0), lw=1)
    ax.plot(apo[:, 0], apo[:, 1], lw=1, label="apo")
    if highlight:
        ax.axvspan(*highlight, color="grey", alpha=0.15, label=f"residues {highlight[0]}-{highlight[1]}")
    ax.set(xlabel="Residue", ylabel="C-alpha RMSF (nm)", title="Flexibility with and without benzene")
    ax.legend(fontsize=8)
    plot_save(fig, out / "rmsf_apo_vs_holo.png")


# =========================================================================== #
# Level 3A - MM/GBSA with gmx_MMPBSA
# =========================================================================== #
MMPBSA_IN = """&general
sys_name="{name}", startframe={start}, endframe={end}, interval={interval},
temperature={temp}, verbose=2,
/
&gb
igb=5, saltcon={salt},
/
"""
MMPBSA_PB = """&pb
istrng={salt}, fillratio=4.0,
/
"""
MMPBSA_DECOMP = """&decomp
idecomp=2, dec_verbose=0, print_res="within 4",
/
"""


def run_mmpbsa(cfg: Config, interval: int = 10, start: int = 1, end: int = 9999999,
               pb: bool = False, decomp: bool = True, workname: str = "L3_mmpbsa") -> dict:
    """MM/GBSA (and optionally MM/PBSA) on the production trajectory of `cfg`.
    Needs gmx_MMPBSA installed in the GROMACS environment (see the notebook)."""
    banner("Level 3A: MM/GBSA with gmx_MMPBSA")
    P, R = runner_for(cfg)
    tpr, xtc, ndx = P.md / "md.tpr", P.ana / "md_fit.xtc", P.ana / "index.ndx"
    for f in (tpr, xtc, ndx):
        if not f.exists():
            raise FileNotFoundError(f"{f} missing - run the Level 1 pipeline for '{cfg.name}' first")
    if not shutil.which("gmx_MMPBSA"):
        raise RuntimeError("gmx_MMPBSA not found - run the install cell of the Level 3 notebook")
    w = stage_dir(cfg, workname)
    for f in list(P.sys.glob("*.itp")) + [P.sys / "topol.top"]:
        shutil.copy(f, w / f.name)
    groups = mp.read_ndx_names(ndx)
    rec, lig = groups.index("Protein"), groups.index(LIG)
    text = MMPBSA_IN.format(name=cfg.name.replace("/", "_"), start=start, end=end, interval=interval,
                            temp=cfg.temperature, salt=cfg.salt_conc)
    if pb:
        text += MMPBSA_PB.format(salt=cfg.salt_conc)
    if decomp:
        text += MMPBSA_DECOMP
    (w / "mmpbsa.in").write_text(text)
    res = w / "FINAL_RESULTS_MMPBSA.dat"
    if res.exists() and not cfg.overwrite:
        print("    already done - skipping (set overwrite=True to redo)")
    else:
        cmd = ["gmx_MMPBSA", "-O", "-i", "mmpbsa.in", "-cs", tpr, "-ci", ndx, "-cg", rec, lig,
               "-ct", xtc, "-cp", "topol.top", "-o", "FINAL_RESULTS_MMPBSA.dat",
               "-eo", "FINAL_RESULTS_MMPBSA.csv", "-nogui"]
        if decomp:
            cmd += ["-do", "FINAL_DECOMP_MMPBSA.dat", "-deo", "FINAL_DECOMP_MMPBSA.csv"]
        R.run(cmd, "L3_mmpbsa", cwd=w)
    out = parse_mmpbsa(res)
    if decomp and (w / "FINAL_DECOMP_MMPBSA.dat").exists():
        out["decomp"] = parse_decomp(w / "FINAL_DECOMP_MMPBSA.dat")
        plot_decomp(out["decomp"], w / "decomposition_top_residues.png")
    (w / "mmpbsa_summary.json").write_text(json.dumps(out, indent=2))
    for k, v in out.items():
        if k != "decomp":
            print(f"    {k}: {v['mean']:.2f} ± {v['sd']:.2f} kcal/mol (SD over frames)")
    return out


def parse_mmpbsa(path: Path) -> dict:
    """Pull the Delta TOTAL lines (one per method) out of FINAL_RESULTS_MMPBSA.dat."""
    text = Path(path).read_text(errors="replace")
    out, method = {}, "GB"
    for line in text.splitlines():
        if "GENERALIZED BORN" in line.upper():
            method = "GB"
        elif "POISSON BOLTZMANN" in line.upper():
            method = "PB"
        m = re.match(r"\s*(?:Δ|DELTA\s*)TOTAL\s+(-?\d+\.\d+)\s+(\d+\.\d+)(?:\s+(\d+\.\d+))?", line)
        if m:
            sd = float(m.group(3) or m.group(2))
            out[f"dG_bind_{method}"] = {"mean": float(m.group(1)), "sd": sd, "units": "kcal/mol"}
    if not out:
        raise RuntimeError(f"No ΔTOTAL line found in {path}; open the file to see what went wrong")
    return out


def parse_decomp(path: Path) -> list:
    """Per-residue TOTAL contributions from the DELTAS block of FINAL_DECOMP_MMPBSA.dat."""
    lines = Path(path).read_text(errors="replace").splitlines()
    try:
        i = next(i for i, l in enumerate(lines) if l.strip().upper().startswith("DELTAS"))
    except StopIteration:
        return []
    out = []
    for l in lines[i + 1:]:
        pairs = re.findall(r"(-?\d+\.\d+)\s*\+/-\s*(\d+\.\d+)", l)
        if not pairs:
            if out and not l.strip():
                break
            continue
        label = re.split(r"[|,]", l)[0].strip()
        if label:
            out.append({"residue": label, "total": float(pairs[-1][0]), "sd": float(pairs[-1][1])})
    return out


def plot_decomp(rows, path: Path, top: int = 12):
    if not rows:
        return
    import matplotlib.pyplot as plt
    rows = sorted(rows, key=lambda r: r["total"])[:top]
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(rows) + 1.2))
    ax.barh([r["residue"] for r in rows][::-1], [r["total"] for r in rows][::-1],
            xerr=[r["sd"] for r in rows][::-1], color="tab:blue")
    ax.set(xlabel="Contribution (kcal/mol)", title="Most favourable residues (MM/GBSA decomposition)")
    plot_save(fig, path)


# =========================================================================== #
# Level 3B - alchemical hydration free energy
# =========================================================================== #
COUL = (0.0, 0.25, 0.5, 0.75, 1.0)
VDW = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0)


def lambda_schedule(coul=COUL, vdw=VDW):
    """Charges are switched off first (vdW kept on), then van der Waals.
    0 = fully coupled, 1 = fully decoupled."""
    c = list(coul) + [1.0] * len(vdw)
    v = [0.0] * len(coul) + list(vdw)
    return c, v


def fe_block(state: int, coul, vdw, molname: str, nstdhdl: int = 100) -> str:
    fmt = lambda xs: " ".join(f"{x:.2f}" for x in xs)
    return f"""; ---- free energy ----
free-energy         = yes
init-lambda-state   = {state}
coul-lambdas        = {fmt(coul)}
vdw-lambdas         = {fmt(vdw)}
calc-lambda-neighbors = -1
couple-moltype      = {molname}
couple-lambda0      = vdw-q
couple-lambda1      = none
couple-intramol     = no
sc-alpha            = 0.5
sc-power            = 1
sc-sigma            = 0.3
nstdhdl             = {nstdhdl}
separate-dhdl-file  = yes
dhdl-derivatives    = yes
"""


def _sd_mdp(cfg: Config, nsteps: int, pcoupl: bool, continuation: bool) -> str:
    """Langevin (sd) dynamics, recommended for free energy windows."""
    m = mp._dyn(cfg, nsteps, max(1, mp._steps(cfg, 1.0)), False, pcoupl, continuation)
    m = set_mdp(m, "integrator", "sd")
    m = set_mdp(m, "tcoupl", "no")              # sd has its own thermostat
    m = set_mdp(m, "tau_t", "1.0")
    m = set_mdp(m, "nstxout-compressed", 0)
    return m


def hydration_topology(cfg: Config, molname: str) -> str:
    return (f'#include "{cfg.forcefield}.ff/forcefield.itp"\n'
            f'#include "{LIG}_atomtypes.itp"\n'
            f'#include "{LIG}.itp"\n'
            f'#include "{cfg.forcefield}.ff/{cfg.water}.itp"\n\n'
            f"[ system ]\nLigand in water\n\n[ molecules ]\n{molname:<20}1\n")


def run_hydration_fe(cfg: Config, coul=COUL, vdw=VDW, equil_ps: float = 100, prod_ns: float = 1.0,
                     box_nm: float = 1.2, windows: list | None = None) -> Path:
    """Decouple the ligand from water window by window. Uses the ligand parameters
    built in Level 1. Each window: EM -> NPT -> production (sd integrator)."""
    banner("Level 3B: alchemical hydration free energy")
    Pb = Paths(cfg)
    for f in (f"{LIG}.itp", f"{LIG}_atomtypes.itp", f"{LIG}.gro"):
        if not (Pb.sys / f).exists():
            raise FileNotFoundError(f"{Pb.sys / f} missing - run Level 1 for '{cfg.name}' first")
    molname = (Pb.lig / "molname.txt").read_text().strip()
    hcfg = replace(cfg, name=f"{cfg.name}/L3_hydration")
    P, R = runner_for(hcfg)
    setup = P.sys                    # L3_hydration/03_system holds the solvated ligand
    for f in (f"{LIG}.itp", f"{LIG}_atomtypes.itp", f"{LIG}.gro"):
        shutil.copy(Pb.sys / f, setup / f)
    if not (setup / "solvated.gro").exists() or cfg.overwrite:
        (setup / "topol.top").write_text(hydration_topology(cfg, molname))
        R.gmx(["editconf", "-f", f"{LIG}.gro", "-o", "boxed.gro", "-bt", "cubic", "-d", box_nm, "-c"],
              "L3_hyd_setup", cwd=setup)
        R.gmx(["solvate", "-cp", "boxed.gro", "-cs", "spc216.gro", "-o", "solvated.gro", "-p", "topol.top"],
              "L3_hyd_setup", cwd=setup)
    c, v = lambda_schedule(coul, vdw)
    (P.root / "lambdas.json").write_text(json.dumps({"coul": c, "vdw": v}))
    todo = windows if windows is not None else range(len(c))
    for i in todo:
        wd = P.root / f"lambda_{i:02d}"
        banner(f"window {i} of {len(c) - 1}: coul-lambda {c[i]:.2f}, vdw-lambda {v[i]:.2f}")
        em = mp.mdp_em(cfg) + fe_block(i, c, v, molname, nstdhdl=0)
        run_md(hcfg, P, R, wd, "em", em, setup / "solvated.gro", stage="em")
        npt = _sd_mdp(cfg, mp._steps(cfg, equil_ps), True, False) + fe_block(i, c, v, molname, nstdhdl=0)
        run_md(hcfg, P, R, wd, "npt", npt, wd / "em.gro")
        prod = _sd_mdp(cfg, mp._steps(cfg, prod_ns * 1000), True, True) + fe_block(i, c, v, molname)
        run_md(hcfg, P, R, wd, "prod", prod, wd / "npt.gro", cpt=wd / "npt.cpt",
               extra=["-dhdl", "dhdl.xvg"])
    return P.root


def analyse_hydration_fe(cfg: Config, skip_ps: float = 100, experiment_kcal: float | None = -0.86):
    """MBAR with alchemlyb. Returns dG_hydration (kcal/mol) and its error."""
    import alchemlyb
    import matplotlib.pyplot as plt
    from alchemlyb.estimators import MBAR
    from alchemlyb.parsing.gmx import extract_u_nk
    from alchemlyb.preprocessing.subsampling import decorrelate_u_nk, slicing
    root = Paths(replace(cfg, name=f"{cfg.name}/L3_hydration")).root
    files = sorted(root.glob("lambda_*/dhdl.xvg"))
    if not files:
        raise FileNotFoundError(f"No dhdl.xvg files under {root} - run run_hydration_fe first")
    T = cfg.temperature
    u_nk = [decorrelate_u_nk(slicing(extract_u_nk(str(f), T=T), lower=skip_ps)) for f in files]
    mbar = MBAR().fit(alchemlyb.concat(u_nk))
    dg_dec_kT = float(mbar.delta_f_.iloc[0, -1])
    err_kT = float(mbar.d_delta_f_.iloc[0, -1])
    kT = KB_KCAL * T
    dg_hyd, err = -dg_dec_kT * kT, err_kT * kT
    out = {"dG_decouple_kcal": dg_dec_kT * kT, "dG_hydration_kcal": dg_hyd, "error_kcal": err,
           "n_windows": len(files), "experiment_kcal": experiment_kcal}
    (root / "hydration_result.json").write_text(json.dumps(out, indent=2))
    try:
        from alchemlyb.visualisation import plot_mbar_overlap_matrix
        ax = plot_mbar_overlap_matrix(mbar.overlap_matrix)
        ax.figure.savefig(root / "overlap_matrix.png", dpi=150)
        plt.show()
    except Exception as e:                     # plotting is a bonus, never fatal
        print("(overlap plot skipped:", e, ")")
    print(f"Hydration free energy: {dg_hyd:.2f} ± {err:.2f} kcal/mol"
          + (f"   (experiment {experiment_kcal} kcal/mol)" if experiment_kcal is not None else ""))
    return out


# =========================================================================== #
# Level 4 - steered MD
# =========================================================================== #
def smd_config(base: Config, box_distance: float = 2.0) -> Config:
    """Same system in a larger box so the ligand can be pulled into bulk water."""
    big = replace(base, name=f"{base.name}_bigbox", box_distance=box_distance, prod_ns=0,
                  launch_chimerax=False)
    reuse_inputs(base, big)          # skip download and ligand parameterisation
    return big


def _moleculetype_files(sysdir: Path):
    """(file, posre include name) for every protein moleculetype with a POSRES block."""
    out = []
    for f in [sysdir / "topol.top", *sorted(sysdir.glob("topol_*.itp"))]:
        if not f.exists():
            continue
        text = f.read_text()
        m = re.search(r'#ifdef POSRES\s*\n\s*#include\s+"([^"]+)"\s*\n\s*#endif', text)
        if m and "[ atoms ]" in text.replace("[atoms]", "[ atoms ]"):
            out.append((f, m.group(1)))
    return out


def add_ca_posres(sysdir: Path, k: float = 500.0) -> list[Path]:
    """Write posre_CA files (C-alpha only, force constant k kJ/mol/nm^2) and add
    `#ifdef POSRES_CA` blocks next to the existing POSRES blocks. Idempotent."""
    written = []
    for f, posre in _moleculetype_files(sysdir):
        text = f.read_text()
        ca_name = posre.replace(".itp", "_CA.itp")
        if f'#include "{ca_name}"' in text:
            written.append(sysdir / ca_name)
            continue
        # C-alpha atom numbers of this moleculetype
        sec, ca = None, []
        for line in text.splitlines():
            m = re.match(r"\s*\[\s*(\w+)\s*\]", line)
            if m:
                sec = m.group(1).lower()
                continue
            body = line.split(";")[0].split()
            if sec == "atoms" and len(body) >= 5 and body[4] == "CA":
                ca.append(int(body[0]))
        if not ca:
            continue
        lines = ["; C-alpha position restraints", "[ position_restraints ]", ";  ai  funct  fcx  fcy  fcz"]
        lines += [f"{i:6d}     1  {k:g}  {k:g}  {k:g}" for i in ca]
        (sysdir / ca_name).write_text("\n".join(lines) + "\n")
        block = f'#ifdef POSRES\n#include "{posre}"\n#endif'
        new = block + f'\n\n; C-alpha restraints for pulling\n#ifdef POSRES_CA\n#include "{ca_name}"\n#endif'
        f.write_text(re.sub(r'#ifdef POSRES\s*\n\s*#include\s+"' + re.escape(posre) + r'"\s*\n\s*#endif',
                            lambda _: new, text, count=1))
        written.append(sysdir / ca_name)
    if not written:
        raise RuntimeError(f"No protein position-restraint block found in {sysdir}")
    return written


def pull_geometry(gro: Path):
    """Reference group (protein C-alpha), ligand group, pulling direction (protein centre ->
    ligand centre) and a pbc atom for the reference group, all from one .gro."""
    atoms, box = read_gro(gro)
    prot = protein_atoms(atoms)
    ca = [a for a in prot if a["name"] == "CA"]
    lig = [a for a in atoms if a["resname"] == LIG]
    if not ca or not lig:
        raise RuntimeError("Need protein C-alpha atoms and a ligand (resname LIG) in " + str(gro))
    c_ca = centre_pbc([a["xyz"] for a in ca], box)
    c_lig = centre_pbc([a["xyz"] for a in lig], box)
    vec = min_image([c_lig[i] - c_ca[i] for i in range(3)], box)
    dist = math.sqrt(sum(x * x for x in vec))
    unit = [x / dist for x in vec]
    pbcatom = min(ca, key=lambda a: sum(d * d for d in min_image(
        [a["xyz"][i] - c_ca[i] for i in range(3)], box)))["index"]
    half_box = 0.5 * min(math.sqrt(sum(x * x for x in row)) for row in box_matrix(box))
    return {"ref": [a["index"] for a in ca], "lig": [a["index"] for a in lig], "vec": unit,
            "start_nm": dist, "pbcatom": pbcatom, "half_box_nm": half_box}


def pull_block(g: dict, rate: float, k: float, nst: int = 50, start: bool = True, init: float = 0.0) -> str:
    v = " ".join(f"{x:.4f}" for x in g["vec"])
    return f"""; ---- COM pulling ----
pull                    = yes
pull-ncoords            = 1
pull-ngroups            = 2
pull-group1-name        = Pull_ref
pull-group2-name        = Pull_lig
pull-group1-pbcatom     = {g['pbcatom']}
pull-pbc-ref-prev-step-com = yes
pull-coord1-groups      = 1 2
pull-coord1-type        = umbrella
pull-coord1-geometry    = direction
pull-coord1-vec         = {v}
pull-coord1-start       = {'yes' if start else 'no'}
pull-coord1-init        = {init}
pull-coord1-rate        = {rate}
pull-coord1-k           = {k}
pull-nstxout            = {nst}
pull-nstfout            = {nst}
"""


def _pull_mdp(cfg: Config, nsteps: int, g: dict, rate: float, k: float, seed: int | None,
              xtc_ps: float = 1.0) -> str:
    m = mp._dyn(cfg, nsteps, max(1, mp._steps(cfg, xtc_ps)), False, True, seed is None)
    m = set_define(m, "-DPOSRES_CA")
    if seed is not None:
        m = set_mdp(m, "gen_seed", seed)
    return m + pull_block(g, rate, k)


def run_smd(cfg_big: Config, n_pulls: int = 5, rate: float = 0.005, k: float = 1000.0,
            pull_nm: float = 2.0, ca_k: float = 500.0) -> Path:
    """Build/equilibrate the big-box system, then pull the ligand out n_pulls times,
    each with new random velocities. rate in nm/ps, k in kJ/mol/nm^2."""
    banner("Level 4: steered MD")
    P, R, _ = mp.prepare_equilibrated(cfg_big)
    add_ca_posres(P.sys, ca_k)
    w = stage_dir(cfg_big, "L4_smd")
    g = pull_geometry(P.npt / "npt.gro")
    if g["start_nm"] + pull_nm > 0.98 * g["half_box_nm"]:
        new = max(0.5, 0.98 * g["half_box_nm"] - g["start_nm"])
        print(f"    pull distance reduced from {pull_nm} to {new:.2f} nm to stay within half the box")
        pull_nm = new
    ndx = w / "pull.ndx"
    write_ndx(ndx, {"System": list(range(1, len(read_gro(P.npt / 'npt.gro')[0]) + 1)),
                    "Pull_ref": g["ref"], "Pull_lig": g["lig"]})
    info = {**{k_: v for k_, v in g.items() if k_ not in ("ref", "lig")}, "rate_nm_per_ps": rate,
            "k_kJ_mol_nm2": k, "pull_nm": pull_nm, "n_pulls": n_pulls, "ca_restraint_k": ca_k}
    (w / "smd_settings.json").write_text(json.dumps(info, indent=2))
    print(f"    start distance {g['start_nm']:.2f} nm, pulling {pull_nm:.2f} nm at {rate} nm/ps "
          f"({pull_nm / rate:.0f} ps per pull)")
    nsteps = mp._steps(cfg_big, pull_nm / rate)
    for i in range(1, n_pulls + 1):
        banner(f"pull {i} of {n_pulls}")
        run_md(cfg_big, P, R, w / f"pull{i}", "pull", _pull_mdp(cfg_big, nsteps, g, rate, k, 2000 + i),
               P.npt / "npt.gro", ref=P.npt / "npt.gro", ndx=ndx,
               extra=["-px", "pullx.xvg", "-pf", "pullf.xvg"])
    return w


def jarzynski(works, T: float):
    """Jarzynski free energy (same units as works, kJ/mol) for each time point.
    works: array (n_pulls, n_points)."""
    import numpy as np
    beta = 1.0 / (KB_KJ * T)
    W = np.asarray(works, float)
    a = -beta * W
    amax = a.max(0)
    lse = amax + np.log(np.exp(a - amax).mean(0))
    jar = -lse / beta
    cum2 = W.mean(0) - beta * W.var(0) / 2           # second-order cumulant estimate
    return jar, cum2


def analyse_smd(cfg_big: Config):
    """Force vs displacement, work vs pulling distance, Jarzynski estimate."""
    import numpy as np
    import matplotlib.pyplot as plt
    w = Paths(cfg_big).root / "L4_smd"
    s = json.loads((w / "smd_settings.json").read_text())
    rate = s["rate_nm_per_ps"]
    pulls = sorted(w.glob("pull*/pullf.xvg"), key=lambda p: int(p.parent.name[4:]))
    if not pulls:
        raise FileNotFoundError("No pulls found - run run_smd first")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    works, xs, rows = [], None, []
    for f in pulls:
        F = xvg_array(f)
        X = xvg_array(f.parent / "pullx.xvg")
        n = min(len(F), len(X))
        t, force, x = F[:n, 0], F[:n, 1], X[:n, 1]
        disp = rate * (t - t[0])                        # spring position relative to start (nm)
        dt = np.diff(t, prepend=t[0])
        W = np.cumsum(force * rate * dt)                # kJ/mol
        a1.plot(x - x[0], force, lw=0.7, label=f.parent.name)
        a2.plot(disp, W, lw=0.8, label=f.parent.name)
        works.append(W)
        xs = disp if xs is None or len(disp) < len(xs) else xs
        rows.append({"pull": f.parent.name, "peak_force_kJ_mol_nm": float(force.max()),
                     "final_work_kJ_mol": float(W[-1])})
    m = min(len(x) for x in works)
    W = np.array([x[:m] for x in works])
    jar, cum2 = jarzynski(W, cfg_big.temperature)
    a2.plot(xs[:m], jar, "k-", lw=2, label="Jarzynski")
    a2.plot(xs[:m], cum2, "k--", lw=1.2, label="2nd-order cumulant")
    a1.set(xlabel="Ligand displacement along pull (nm)", ylabel="Force (kJ/mol/nm)", title="Pulling force")
    a2.set(xlabel="Spring displacement (nm)", ylabel="Work (kJ/mol)", title="Work and Jarzynski estimate")
    a1.legend(fontsize=7)
    a2.legend(fontsize=7)
    plot_save(fig, w / "smd_force_work.png")
    out = {"pulls": rows, "jarzynski_final_kJ_mol": float(jar[-1]),
           "cumulant_final_kJ_mol": float(cum2[-1]), "mean_work_final_kJ_mol": float(W[:, -1].mean())}
    (w / "smd_summary.json").write_text(json.dumps(out, indent=2))
    for r in rows:
        print(f"    {r['pull']}: peak force {r['peak_force_kJ_mol_nm']:.0f} kJ/mol/nm, "
              f"work {r['final_work_kJ_mol']:.1f} kJ/mol")
    print(f"    mean work {out['mean_work_final_kJ_mol']:.1f}, Jarzynski {out['jarzynski_final_kJ_mol']:.1f} "
          f"kJ/mol (divide by 4.184 for kcal/mol)")
    return out


# =========================================================================== #
# Level 5A - umbrella sampling + WHAM
# =========================================================================== #
def choose_windows(times, values, spacing: float = 0.1, frame_dt: float = 1.0):
    """Pick (time, value) pairs about `spacing` nm apart along the pull coordinate, using
    only times at which a trajectory frame exists (multiples of frame_dt ps)."""
    import numpy as np
    t, x = np.asarray(times, float), np.asarray(values, float)
    ok = np.isclose((t - t[0]) / frame_dt, np.round((t - t[0]) / frame_dt), atol=1e-3)
    t, x = t[ok], x[ok]
    targets = np.arange(x[0], x.max() + 1e-9, spacing)
    picks = []
    for tg in targets:
        i = int(np.argmin(np.abs(x - tg)))
        if not picks or abs(x[i] - picks[-1][1]) > spacing / 2:
            picks.append((float(t[i]), float(x[i])))
    return picks


def run_umbrella(cfg_big: Config, pull: int = 1, spacing: float = 0.1, k: float = 1000.0,
                 equil_ps: float = 100, prod_ns: float = 1.0, windows: list | None = None) -> Path:
    """Umbrella windows from snapshots of one Level 4 pull."""
    banner("Level 5A: umbrella sampling")
    P, R = runner_for(cfg_big)
    smd = P.root / "L4_smd" / f"pull{pull}"
    if not (smd / "pull.gro").exists():
        raise FileNotFoundError(f"{smd} incomplete - finish Level 4 first")
    w = stage_dir(cfg_big, "L5_umbrella")
    g = pull_geometry(P.npt / "npt.gro")
    ndx = P.root / "L4_smd" / "pull.ndx"
    X = xvg_array(smd / "pullx.xvg")
    picks = choose_windows(X[:, 0], X[:, 1], spacing)
    (w / "windows.json").write_text(json.dumps([{"window": i, "time_ps": t, "coord_nm": x}
                                                for i, (t, x) in enumerate(picks)], indent=2))
    print(f"    {len(picks)} windows from {picks[0][1]:.2f} to {picks[-1][1]:.2f} nm")
    todo = windows if windows is not None else range(len(picks))
    for i in todo:
        t, x = picks[i]
        wd = w / f"win_{i:02d}"
        wd.mkdir(exist_ok=True)
        conf = wd / "start.gro"
        if not conf.exists():
            R.gmx(["trjconv", "-s", smd / "pull.tpr", "-f", smd / "pull.xtc", "-o", conf, "-dump", t],
                  "L5_umbrella_frames", stdin="0\n")
        banner(f"window {i} of {len(picks) - 1}: {x:.2f} nm")
        eq = mp._dyn(cfg_big, mp._steps(cfg_big, equil_ps), max(1, mp._steps(cfg_big, 1.0)), False, True, False)
        eq = set_define(eq, "-DPOSRES_CA") + pull_block(g, 0.0, k)
        run_md(cfg_big, P, R, wd, "npt", eq, conf, ref=conf, ndx=ndx)
        pr = mp._dyn(cfg_big, mp._steps(cfg_big, prod_ns * 1000), max(1, mp._steps(cfg_big, 10.0)),
                     False, True, True)
        pr = set_define(pr, "-DPOSRES_CA") + pull_block(g, 0.0, k)
        run_md(cfg_big, P, R, wd, "umbrella", pr, wd / "npt.gro", cpt=wd / "npt.cpt", ref=conf, ndx=ndx,
               extra=["-px", "pullx.xvg", "-pf", "pullf.xvg"])
    return w


def run_wham(cfg_big: Config, skip_ps: float = 100, n_bootstrap: int = 50, units: str = "kCal"):
    """gmx wham over all finished windows; plots PMF (with bootstrap error) and histograms."""
    import numpy as np
    import matplotlib.pyplot as plt
    P, R = runner_for(cfg_big)
    w = Paths(cfg_big).root / "L5_umbrella"
    wins = sorted(d for d in w.glob("win_*") if (d / "umbrella.gro").exists())
    if len(wins) < 3:
        raise RuntimeError(f"Only {len(wins)} finished windows - need at least 3")
    (w / "tpr-files.dat").write_text("\n".join(str(d / "umbrella.tpr") for d in wins) + "\n")
    (w / "pullf-files.dat").write_text("\n".join(str(d / "pullf.xvg") for d in wins) + "\n")
    R.gmx(["wham", "-it", "tpr-files.dat", "-if", "pullf-files.dat", "-o", "profile.xvg",
           "-hist", "histo.xvg", "-unit", units, "-b", skip_ps, "-temp", cfg_big.temperature,
           "-nBootstrap", n_bootstrap, "-bsres", "bsResult.xvg", "-bsprof", "bsProfs.xvg"],
          "L5_wham", cwd=w)
    prof = xvg_array(w / "profile.xvg")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    unit = "kcal/mol" if units.lower() == "kcal" else "kJ/mol" if units.lower() == "kj" else "kT"
    a1.plot(prof[:, 0], prof[:, 1], "k-", lw=1.5, label="PMF")
    if (w / "bsResult.xvg").exists():
        bs = xvg_array(w / "bsResult.xvg")
        if bs.shape[1] >= 3:
            a1.fill_between(bs[:, 0], bs[:, 1] - bs[:, 2], bs[:, 1] + bs[:, 2], alpha=0.3,
                            label="bootstrap ± SD")
    a1.set(xlabel="Pull coordinate (nm)", ylabel=f"PMF ({unit})", title="Potential of mean force")
    a1.legend(fontsize=8)
    h = xvg_array(w / "histo.xvg")
    for j in range(1, h.shape[1]):
        a2.plot(h[:, 0], h[:, j], lw=0.8)
    a2.set(xlabel="Pull coordinate (nm)", ylabel="Count", title="Window histograms (must overlap)")
    plot_save(fig, w / "pmf.png")
    depth = float(prof[:, 1].max() - prof[:, 1].min())
    print(f"    {len(wins)} windows; PMF range {depth:.2f} {unit} (max minus min)")
    return {"windows": len(wins), "pmf_range": depth, "units": unit}


# =========================================================================== #
# Level 5B - metadynamics with PLUMED
# =========================================================================== #
def plumed_available(cfg: Config) -> bool:
    import os
    kern = os.environ.get("PLUMED_KERNEL", "")
    h = subprocess.run([cfg.gmx, "mdrun", "-h"], capture_output=True, text=True)
    ok_flag = "-plumed" in (h.stdout + h.stderr)
    ok_kern = bool(kern) and Path(kern).exists()
    print(f"    gmx mdrun -plumed option: {'yes' if ok_flag else 'NO'};  PLUMED_KERNEL: {kern or 'not set'}"
          f" ({'found' if ok_kern else 'MISSING'})")
    return ok_flag and ok_kern


def dipeptide_config(base: Config | None = None, **kw) -> Config:
    base = base or Config()
    d = dict(name="ala_dipeptide", pdb_id=None, pdb_file=None, ligand_resname=None, ligand_file=None,
             fix_protein=False, box_distance=1.0, prod_ns=0, launch_chimerax=False)
    d.update(kw)
    return replace(base, **d)


LEAP_DIPEPTIDE = """source leaprc.protein.ff14SB
m = sequence { ACE ALA NME }
savepdb m ala2_leap.pdb
quit
"""


def build_dipeptide(cfg: Config):
    """Make ACE-ALA-NME with tleap (AmberTools), then build/equilibrate it with the base pipeline."""
    banner("Level 5B: alanine dipeptide")
    root = Paths(cfg).root
    root.mkdir(parents=True, exist_ok=True)
    pdb = root / "ala2_leap.pdb"
    if not pdb.exists():
        (root / "leap.in").write_text(LEAP_DIPEPTIDE)
        r = subprocess.run(["tleap", "-f", "leap.in"], cwd=root, capture_output=True, text=True)
        (root / "leap.log.txt").write_text(r.stdout + r.stderr)
        if not pdb.exists():
            raise RuntimeError(f"tleap failed - see {root / 'leap.log.txt'}")
    cfg.pdb_file = str(pdb)
    return mp.prepare_equilibrated(cfg)


def dipeptide_torsions(gro: Path):
    """1-based atom numbers for phi (C-N-CA-C) and psi (N-CA-C-N) of the alanine."""
    atoms, _ = read_gro(gro)
    resnrs = sorted({a["resnr"] for a in atoms if a["resname"] in ("ACE", "ALA", "NME")})
    ace, ala, nme = resnrs[:3]
    pick = lambda r, n: next(a["index"] for a in atoms if a["resnr"] == r and a["name"] == n)
    phi = [pick(ace, "C"), pick(ala, "N"), pick(ala, "CA"), pick(ala, "C")]
    psi = [pick(ala, "N"), pick(ala, "CA"), pick(ala, "C"), pick(nme, "N")]
    return phi, psi


def dipeptide_plumed(phi, psi, metad: bool = True, pace: int = 500, height: float = 1.2,
                     sigma: float = 0.35, biasfactor: float = 6, temp: float = 300) -> str:
    j = lambda xs: ",".join(map(str, xs))
    s = f"phi: TORSION ATOMS={j(phi)}\npsi: TORSION ATOMS={j(psi)}\n"
    if metad:
        s += (f"metad: METAD ARG=phi,psi PACE={pace} HEIGHT={height} SIGMA={sigma},{sigma} "
              f"BIASFACTOR={biasfactor} TEMP={temp} FILE=HILLS "
              f"GRID_MIN=-pi,-pi GRID_MAX=pi,pi GRID_BIN=150,150\n")
        s += "PRINT ARG=phi,psi,metad.bias STRIDE=500 FILE=COLVAR\n"
    else:
        s += "PRINT ARG=phi,psi STRIDE=500 FILE=COLVAR\n"
    return s


def run_dipeptide_metad(cfg: Config, ns: float = 10.0, unbiased: bool = True, **metad_kw) -> Path:
    if not plumed_available(cfg):
        raise RuntimeError("GROMACS cannot use PLUMED here - see the install cell of the notebook")
    P, R, _ = build_dipeptide(cfg)
    phi, psi = dipeptide_torsions(P.npt / "npt.gro")
    w = stage_dir(cfg, "L5_metad")
    (w / "torsions.json").write_text(json.dumps({"phi": phi, "psi": psi}))
    nsteps = mp._steps(cfg, ns * 1000)
    mdp = mp._dyn(cfg, nsteps, max(1, mp._steps(cfg, 1.0)), False, True, True)
    runs = (["unbiased"] if unbiased else []) + ["metad"]
    for r in runs:
        banner(f"{r} run, {ns} ns")
        run_md(cfg, P, R, w / r, "md", mdp, P.npt / "npt.gro", cpt=P.npt / "npt.cpt",
               plumed=dipeptide_plumed(phi, psi, metad=(r == "metad"), temp=cfg.temperature, **metad_kw))
    return w


def read_colvar(path: Path):
    import numpy as np
    names, rows = [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("#! FIELDS"):
            names = line.split()[2:]
        elif line.strip() and not line.startswith("#"):
            try:
                rows.append([float(x) for x in line.split()])
            except ValueError:
                pass
    a = np.array(rows)
    return {n: a[:, i] for i, n in enumerate(names) if i < a.shape[1]}


def sum_hills(workdir: Path, ndim: int = 2, stride: int | None = None, extra: list | None = None):
    cmd = ["plumed", "sum_hills", "--hills", "HILLS", "--mintozero"]
    if stride:
        cmd += ["--stride", str(stride), "--outfile", "fes_t_"]
    else:
        cmd += ["--outfile", "fes.dat"]
    cmd += list(extra or [])
    r = subprocess.run(cmd, cwd=workdir, capture_output=True, text=True)
    (workdir / "sum_hills.log").write_text(r.stdout + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"plumed sum_hills failed - see {workdir / 'sum_hills.log'}")


def read_fes2d(path: Path):
    import numpy as np
    d = np.loadtxt(path, comments="#")
    xs, ys = np.unique(d[:, 0]), np.unique(d[:, 1])
    F = np.full((len(ys), len(xs)), np.nan)
    ix = np.searchsorted(xs, d[:, 0])
    iy = np.searchsorted(ys, d[:, 1])
    F[iy, ix] = d[:, 2]
    return xs, ys, F


def basin_dF(xs, ys, F, T: float):
    """Free energy of phi>0 relative to phi<0 (kJ/mol), by Boltzmann-summing the FES."""
    import numpy as np
    kT = KB_KJ * T
    p = np.exp(-(F - np.nanmin(F)) / kT)
    p = np.nan_to_num(p)
    pos = p[:, xs > 0].sum()
    neg = p[:, xs <= 0].sum()
    return float(-kT * math.log(pos / neg)) if pos > 0 and neg > 0 else float("nan")


def analyse_dipeptide_metad(cfg: Config, n_snapshots: int = 10):
    import numpy as np
    import matplotlib.pyplot as plt
    w = Paths(cfg).root / "L5_metad"
    m = w / "metad"
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    if (w / "unbiased" / "COLVAR").exists():
        cv = read_colvar(w / "unbiased" / "COLVAR")
        axes[0].scatter(cv["phi"], cv["psi"], s=2, alpha=0.5)
        axes[0].set(xlim=(-math.pi, math.pi), ylim=(-math.pi, math.pi), xlabel="φ (rad)", ylabel="ψ (rad)",
                    title="Unbiased MD: where it went")
    cvm = read_colvar(m / "COLVAR")
    axes[1].scatter(cvm["phi"], cvm["psi"], s=2, alpha=0.5, c=cvm["time"], cmap="viridis")
    axes[1].set(xlim=(-math.pi, math.pi), ylim=(-math.pi, math.pi), xlabel="φ (rad)", ylabel="ψ (rad)",
                title="Metadynamics: where it went")
    sum_hills(m)
    xs, ys, F = read_fes2d(m / "fes.dat")
    cs = axes[2].contourf(xs, ys, np.clip(F, 0, 50), levels=25, cmap="viridis")
    fig.colorbar(cs, ax=axes[2], label="Free energy (kJ/mol)")
    axes[2].set(xlabel="φ (rad)", ylabel="ψ (rad)", title="Free-energy surface")
    plot_save(fig, w / "dipeptide_fes.png")

    # convergence: dF(phi>0 vs phi<0) as hills accumulate
    nh = sum(1 for l in (m / "HILLS").read_text().splitlines() if l.strip() and not l.startswith("#"))
    stride = max(1, nh // n_snapshots)
    sum_hills(m, stride=stride)
    snaps = sorted(m.glob("fes_t*.dat"), key=lambda p: int(re.findall(r"\d+", p.stem)[-1]))
    pace_ps = cfg.dt * 500
    dfs = []
    for i, f in enumerate(snaps):
        xs_, ys_, F_ = read_fes2d(f)
        dfs.append(((i + 1) * stride * pace_ps / 1000, basin_dF(xs_, ys_, F_, cfg.temperature)))
    if dfs:
        fig, ax = plt.subplots(figsize=(6, 3.4))
        ax.plot([d[0] for d in dfs], [d[1] for d in dfs], "o-")
        ax.set(xlabel="Simulation time (ns)", ylabel="ΔF(φ>0 − φ<0) (kJ/mol)",
               title="Convergence: should level off")
        plot_save(fig, w / "dipeptide_convergence.png")
    out = {"dF_phi_pos_minus_neg_kJ": dfs[-1][1] if dfs else None, "convergence": dfs}
    (w / "metad_summary.json").write_text(json.dumps(out, indent=2))
    return out


def ligand_plumed(nprot: int, lig: list, pocket: list, wall_nm: float = 2.5, pace: int = 500,
                  height: float = 1.0, sigma: float = 0.05, biasfactor: float = 10, temp: float = 300) -> str:
    j = lambda xs: ",".join(map(str, xs))
    return (f"WHOLEMOLECULES ENTITY0=1-{nprot} ENTITY1={lig[0]}-{lig[-1]}\n"
            f"lig: COM ATOMS={lig[0]}-{lig[-1]}\n"
            f"pocket: COM ATOMS={j(pocket)}\n"
            f"d: DISTANCE ATOMS=lig,pocket\n"
            f"uwall: UPPER_WALLS ARG=d AT={wall_nm} KAPPA=2000.0\n"
            f"metad: METAD ARG=d PACE={pace} HEIGHT={height} SIGMA={sigma} BIASFACTOR={biasfactor} "
            f"TEMP={temp} FILE=HILLS GRID_MIN=0 GRID_MAX={wall_nm + 1.0} GRID_BIN={int((wall_nm + 1.0) / 0.01)}\n"
            f"PRINT ARG=d,metad.bias,uwall.bias STRIDE=500 FILE=COLVAR\n")


def run_ligand_metad(cfg: Config, ns: float = 20.0, pocket_cutoff_nm: float = 0.8,
                     wall_nm: float = 2.5, **metad_kw) -> Path:
    """Well-tempered metadynamics on the ligand-pocket distance (Level 1 system)."""
    banner("Level 5B (optional): ligand metadynamics")
    if not plumed_available(cfg):
        raise RuntimeError("GROMACS cannot use PLUMED here - see the install cell of the notebook")
    P, R, _ = mp.prepare_equilibrated(cfg)
    atoms, box = read_gro(P.npt / "npt.gro")
    prot = protein_atoms(atoms)
    lig = [a["index"] for a in atoms if a["resname"] == LIG]
    lc = centre_pbc([atoms[i - 1]["xyz"] for i in lig], box)
    pocket = [a["index"] for a in prot if a["name"] == "CA" and
              math.dist([0, 0, 0], min_image([a["xyz"][i] - lc[i] for i in range(3)], box)) < pocket_cutoff_nm]
    if len(pocket) < 3:
        raise RuntimeError("Fewer than 3 C-alpha atoms near the ligand - increase pocket_cutoff_nm")
    if wall_nm > 0.95 * 0.5 * min(math.sqrt(sum(x * x for x in r)) for r in box_matrix(box)):
        print("    WARNING: wall is close to half the box; distances may wrap through the periodic box")
    w = stage_dir(cfg, "L5_metad_ligand")
    (w / "groups.json").write_text(json.dumps({"ligand": lig, "pocket_CA": pocket, "n_protein_atoms": len(prot)}))
    mdp = mp._dyn(cfg, mp._steps(cfg, ns * 1000), max(1, mp._steps(cfg, 10.0)), False, True, True)
    run_md(cfg, P, R, w, "metad", mdp, P.npt / "npt.gro", cpt=P.npt / "npt.cpt",
           plumed=ligand_plumed(len(prot), lig, pocket, wall_nm, temp=cfg.temperature, **metad_kw))
    return w


def analyse_ligand_metad(cfg: Config):
    import numpy as np
    import matplotlib.pyplot as plt
    w = Paths(cfg).root / "L5_metad_ligand"
    cv = read_colvar(w / "COLVAR")
    sum_hills(w)
    d = np.loadtxt(w / "fes.dat", comments="#")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(cv["time"] / 1000, cv["d"], lw=0.6)
    a1.set(xlabel="Time (ns)", ylabel="Ligand-pocket distance (nm)", title="Does the ligand leave?")
    a2.plot(d[:, 0], d[:, 1])
    a2.set(xlabel="Ligand-pocket distance (nm)", ylabel="Free energy (kJ/mol)",
           title="1D free-energy profile (not converged unless it rebinds)")
    plot_save(fig, w / "ligand_metad.png")
    first_exit = next((t for t, x in zip(cv["time"], cv["d"]) if x > 1.5), None)
    print(f"    max distance {cv['d'].max():.2f} nm; first time beyond 1.5 nm: "
          f"{'never' if first_exit is None else f'{first_exit / 1000:.1f} ns'}")
    return {"max_distance_nm": float(cv["d"].max()),
            "first_exit_ns": None if first_exit is None else first_exit / 1000}

In [ ]:
import sys, importlib
sys.path.insert(0, "/content")
import md_pipeline, md_advanced
importlib.reload(md_pipeline); importlib.reload(md_advanced)
from md_pipeline import Config, Paths, run_pipeline, show_in_notebook
import md_advanced as adv

### The system
The same settings as Level 1, so results from Level 1 are reused (finished steps are skipped). If you changed the system name or the output folder in Level 1, change them here too.

In [ ]:
base = Config(
    name="t4l_benzene", pdb_id="181L", ligand_resname="BNZ", ligand_charge=0,
    temperature=300.0, salt_conc=0.15, nvt_ps=100, npt_ps=100, prod_ns=10, traj_interval_ps=10,
    output_root=OUTPUT_ROOT, gmx=f"{ENV}/bin/gmx", use_gpu=True, checkpoint_min=5,
)
print(Paths(base).root)

In [ ]:
big = adv.smd_config(base, box_distance=2.0)   # same big-box system as Level 4
PROD_NS = 1.0          # per window
WINDOWS = None         # e.g. list(range(0, 10)) this session, list(range(10, 25)) next
us_dir = adv.run_umbrella(big, pull=1, spacing=0.1, k=1000.0, prod_ns=PROD_NS, windows=WINDOWS)

## WHAM: the free-energy profile
- **PMF (left):** free energy along the distance. The bound state is the minimum at short distance, and the plateau at large distance is benzene in water. Depth = plateau − minimum. Barriers on the way show where the exit is hard. The shaded band is the bootstrap error.
- **Histograms (right):** each window's sampled range. Neighbours must overlap. A gap means the PMF across it is unreliable: add a window there (`WINDOWS` + a smaller `spacing`) or use a weaker spring.

In [ ]:
pmf = adv.run_wham(big, skip_ps=100, n_bootstrap=50, units="kCal")
pmf

## Exercises
1. Compare the PMF depth with MM/GBSA (Level 3), the Jarzynski estimate (Level 4) and experiment (about −5.2 kcal/mol). The 1D PMF is not yet a standard binding free energy. Corrections for standard state (1 M), orientation and the protein restraints are needed (see the LiveCoMS free-energy best-practices paper).
2. Run WHAM with `skip_ps=500`. Does the profile change? If it does, the windows weren't equilibrated.
3. Extend the windows with the largest bootstrap error and rerun WHAM.